In [750]:
# ============================================================
# PART 1
# Production XGBoost MONEY_OUT Forecasting
# Environment + Configuration
# ============================================================

from __future__ import annotations

import logging
import platform
import random
import warnings

from dataclasses import dataclass
from datetime import datetime
from typing import List, Tuple, Optional

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


# ------------------------------------------------------------
# Logging
# ------------------------------------------------------------

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

logger = logging.getLogger("money_out_forecasting")


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

@dataclass
class ForecastConfig:

    # Input / output
    input_table: str = "financial_bronze"
    output_table: str = "financial_forecast"

    # Source columns
    sponsor_col: str = "SPSR_ID"
    subscriber_col: str = "SBSR_ID"
    date_col: str = "YEAR_MONTH"
    summary_col: str = "SUMMARY_TYPE"
    target_col: str = "AMOUNT"

    # Required summary type
    summary_type: str = "MoneyOut"

    # Forecast horizon
    forecast_horizon: int = 2

    # Reproducibility
    random_seed: int = 42

    # Minimum history
    min_history: int = 3

    # Model
    model_name: str = "XGBoost-MONEY-OUT"

    # Currency target cannot be negative in our expected
    # business scenario.
    enforce_non_negative_forecast: bool = True


CONFIG = ForecastConfig()

logger.info("Forecast configuration initialized.")
logger.info("Input table: %s", CONFIG.input_table)
logger.info("Forecast horizon: %s months", CONFIG.forecast_horizon)




2026-08-30 21:37:26,704 | INFO | Forecast configuration initialized.
2026-08-30 21:37:26,705 | INFO | Input table: financial_bronze
2026-08-30 21:37:26,706 | INFO | Forecast horizon: 2 months


In [751]:
%pip install pandas numpy xgboost scikit-learn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\udayn\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [752]:

import pandas as pd
 
# ----------------------------------------------------------------------
# 1. LOAD DATA
# ----------------------------------------------------------------------
# Replace this block with pd.read_csv("your_file.csv") if reading from disk.
data = {
    "SPSR_ID": ["AG0025"] * 12,
    "SBSR_ID": ["AG0025S100000097"] * 12,
    "YEAR_MONTH": [
        "2025-08-01", "2025-09-01", "2025-10-01",
        "2025-11-01", "2025-12-01", "2026-01-01","2026-02-01",
        "2026-03-01","2026-04-01","2026-05-01","2026-06-01","2026-07-01"

    ],
    
    "AMOUNT": [513.00, 19268.75, 28054.06, 18634.24,
                         32127.32, 42061.26, 55587.25, 68903.75, 82319.75, 95735.75, 109151.75, 122567.75],
    "SUMMARY_TYPE": ["MoneyOut"] * 12
}
df = pd.DataFrame(data)
df["YEAR_MONTH"] = pd.to_datetime(df["YEAR_MONTH"])
df = df.sort_values("YEAR_MONTH").reset_index(drop=True)

display(df)


print("Source columns:")
print(df.columns)

,SPSR_ID,SBSR_ID,YEAR_MONTH,AMOUNT,SUMMARY_TYPE
0,AG0025,AG0025S100000097,2025-08-01,513.00,MoneyOut
1,AG0025,AG0025S100000097,2025-09-01,19268.75,MoneyOut
2,AG0025,AG0025S100000097,2025-10-01,28054.06,MoneyOut
3,AG0025,AG0025S100000097,2025-11-01,18634.24,MoneyOut
4,AG0025,AG0025S100000097,2025-12-01,32127.32,MoneyOut
5,AG0025,AG0025S100000097,2026-01-01,42061.26,MoneyOut
6,AG0025,AG0025S100000097,2026-02-01,55587.25,MoneyOut
7,AG0025,AG0025S100000097,2026-03-01,68903.75,MoneyOut
8,AG0025,AG0025S100000097,2026-04-01,82319.75,MoneyOut
9,AG0025,AG0025S100000097,2026-05-01,95735.75,MoneyOut


Source columns:
Index(['SPSR_ID', 'SBSR_ID', 'YEAR_MONTH', 'AMOUNT', 'SUMMARY_TYPE'], dtype='str')


In [753]:
# ============================================================
# Validate required columns
# ============================================================

required_columns = {
    CONFIG.sponsor_col,
    CONFIG.subscriber_col,
    CONFIG.date_col,
    CONFIG.summary_col,
    CONFIG.target_col
}

actual_columns = set(df.columns)

missing_columns = required_columns - actual_columns

if missing_columns:
    raise ValueError(
        f"Missing required columns: {sorted(missing_columns)}"
    )

logger.info("All required columns are present.")

2026-08-30 21:37:28,670 | INFO | All required columns are present.


In [754]:
# ============================================================
# Filter TOTAL summary
# ============================================================

df_spark = df.filter(
    df[CONFIG.summary_col] == CONFIG.summary_type
)

logger.info(
    "Filtered source to SUMMARY_TYPE = '%s'",
    CONFIG.summary_type
)

print(f"Rows after TOTAL filter: {df_spark.count():}")

2026-08-30 21:37:28,950 | INFO | Filtered source to SUMMARY_TYPE = 'MoneyOut'


Rows after TOTAL filter: Series([], dtype: int64)


In [755]:
display(df )

,SPSR_ID,SBSR_ID,YEAR_MONTH,AMOUNT,SUMMARY_TYPE
0,AG0025,AG0025S100000097,2025-08-01,513.00,MoneyOut
1,AG0025,AG0025S100000097,2025-09-01,19268.75,MoneyOut
2,AG0025,AG0025S100000097,2025-10-01,28054.06,MoneyOut
3,AG0025,AG0025S100000097,2025-11-01,18634.24,MoneyOut
4,AG0025,AG0025S100000097,2025-12-01,32127.32,MoneyOut
5,AG0025,AG0025S100000097,2026-01-01,42061.26,MoneyOut
6,AG0025,AG0025S100000097,2026-02-01,55587.25,MoneyOut
7,AG0025,AG0025S100000097,2026-03-01,68903.75,MoneyOut
8,AG0025,AG0025S100000097,2026-04-01,82319.75,MoneyOut
9,AG0025,AG0025S100000097,2026-05-01,95735.75,MoneyOut


In [756]:
# ============================================================
# Basic inspection
# ============================================================

print("Shape:", df.shape)

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values:")
display(
    df.isna()
      .sum()
      .sort_values(ascending=False)
      .to_frame("missing_count")
)

print("\nDuplicate rows:", df.duplicated().sum())

Shape: (12, 5)

Data types:


,dtype
SPSR_ID,str
SBSR_ID,str
YEAR_MONTH,datetime64[us]
AMOUNT,float64
SUMMARY_TYPE,str



Missing values:


,missing_count
SPSR_ID,0
SBSR_ID,0
YEAR_MONTH,0
AMOUNT,0
SUMMARY_TYPE,0



Duplicate rows: 0


In [757]:
# ============================================================
# Convert MONTH_YEAR
# ============================================================

df[CONFIG.date_col] = pd.to_datetime(
    df[CONFIG.date_col],
    format="%Y-%m",
    errors="coerce"
)

invalid_dates = df[CONFIG.date_col].isna().sum()

if invalid_dates > 0:
    raise ValueError(
        f"Found {invalid_dates:,} invalid MONTH_YEAR values."
    )

# Normalize to month-start
df[CONFIG.date_col] = (
    df[CONFIG.date_col]
    .dt.to_period("M")
    .dt.to_timestamp()
)

logger.info("MONTH_YEAR successfully converted to monthly datetime.")

2026-08-30 21:37:29,634 | INFO | MONTH_YEAR successfully converted to monthly datetime.


In [758]:
# ============================================================
# Validate MONEY_OUT
# ============================================================

df[CONFIG.target_col] = pd.to_numeric(
    df[CONFIG.target_col],
    errors="coerce"
)

missing_target = df[CONFIG.target_col].isna().sum()

if missing_target > 0:
    logger.warning(
        "Found %s rows with missing MONEY_OUT.",
        f"{missing_target:,}"
    )

negative_target = (
    df[CONFIG.target_col] < 0
).sum()

if negative_target > 0:
    logger.warning(
        "Found %s rows with negative MONEY_OUT.",
        f"{negative_target:,}"
    )

In [759]:
# ============================================================
# Duplicate handling
# ============================================================

duplicate_count = df.duplicated().sum()

if duplicate_count > 0:

    logger.warning(
        "Found %s exact duplicate rows.",
        f"{duplicate_count:,}"
    )

    df = df.drop_duplicates().copy()

else:

    logger.info("No exact duplicate rows found.")

2026-08-30 21:37:29,804 | INFO | No exact duplicate rows found.


In [760]:
# ============================================================
# Check business-grain duplicates
# ============================================================

grain_columns = [
    CONFIG.sponsor_col,
    CONFIG.subscriber_col,
    CONFIG.date_col,
    CONFIG.summary_col
]

grain_counts = (
    df.groupby(grain_columns)
      .size()
      .reset_index(name="ROW_COUNT")
)

business_duplicates = grain_counts[
    grain_counts["ROW_COUNT"] > 1
]

print(
    f"Business-grain duplicate groups: "
    f"{len(business_duplicates):,}"
)

if len(business_duplicates) > 0:

    display(
        business_duplicates.head(20)
    )

    logger.warning(
        "Multiple records exist at the expected business grain."
    )

Business-grain duplicate groups: 0


In [761]:
# ============================================================
# Forecasting series analysis
# ============================================================

series_columns = [
    CONFIG.sponsor_col,
    CONFIG.subscriber_col
]

series_count = (
    df[series_columns]
    .drop_duplicates()
    .shape[0]
)

logger.info(
    "Number of Sponsor + Subscriber forecasting series: %s",
    f"{series_count:,}"
)

print(
    "Unique Sponsor + Subscriber series:",
    f"{series_count:,}"
)

2026-08-30 21:37:30,214 | INFO | Number of Sponsor + Subscriber forecasting series: 1


Unique Sponsor + Subscriber series: 1


In [762]:
# ============================================================
# History-length analysis
# ============================================================

history_summary = (
    df.groupby(series_columns)[CONFIG.date_col]
      .nunique()
      .reset_index(name="HISTORY_MONTHS")
)

print("History distribution:")

display(
    history_summary["HISTORY_MONTHS"]
    .describe()
    .to_frame()
)

History distribution:


,HISTORY_MONTHS
count,1.0
mean,12.0
std,NaN
min,12.0
25%,12.0
50%,12.0
75%,12.0
max,12.0


In [763]:
# ============================================================
# History buckets
# ============================================================

def history_bucket(months: int) -> str:

    if months < 3:
        return "<3 months"

    if months <= 5:
        return "3-5 months"

    if months <= 11:
        return "6-11 months"

    return "12+ months"


history_summary["HISTORY_BUCKET"] = (
    history_summary["HISTORY_MONTHS"]
    .apply(history_bucket)
)

display(
    history_summary["HISTORY_BUCKET"]
    .value_counts()
    .to_frame("SERIES_COUNT")
)

,SERIES_COUNT
HISTORY_BUCKET,
12+ months,1


In [764]:
# ============================================================
# Detect missing months
# ============================================================

def find_missing_months(
    group: pd.DataFrame
) -> pd.DataFrame:

    dates = (
        group[CONFIG.date_col]
        .dropna()
        .drop_duplicates()
        .sort_values()
    )

    if len(dates) <= 1:
        return pd.DataFrame()

    expected = pd.date_range(
        start=dates.min(),
        end=dates.max(),
        freq="MS"
    )

    missing = expected.difference(dates)

    if len(missing) == 0:
        return pd.DataFrame()

    return pd.DataFrame({
        CONFIG.sponsor_col: [
            group[CONFIG.sponsor_col].iloc[0]
        ] * len(missing),

        CONFIG.subscriber_col: [
            group[CONFIG.subscriber_col].iloc[0]
        ] * len(missing),

        "MISSING_MONTH": missing
    })


missing_month_results = []

for _, group in df.groupby(series_columns):

    result = find_missing_months(group)

    if not result.empty:
        missing_month_results.append(result)


if missing_month_results:

    missing_months_df = pd.concat(
        missing_month_results,
        ignore_index=True
    )

else:

    missing_months_df = pd.DataFrame()


print(
    "Missing month records:",
    f"{len(missing_months_df):,}"
)

if not missing_months_df.empty:
    display(missing_months_df.head(20))

Missing month records: 0


In [765]:
# ============================================================
# Sort chronologically
# ============================================================

df = (
    df.sort_values(
        by=[
            CONFIG.sponsor_col,
            CONFIG.subscriber_col,
            CONFIG.date_col
        ]
    )
    .reset_index(drop=True)
)

logger.info("Data sorted chronologically by forecasting series.")

2026-08-30 21:37:32,069 | INFO | Data sorted chronologically by forecasting series.


In [766]:
# ============================================================
# Determine forecast cutoff
# ============================================================

latest_month = df[CONFIG.date_col].max()

forecast_month_1 = (
    latest_month + pd.DateOffset(months=1)
)

forecast_month_2 = (
    latest_month + pd.DateOffset(months=2)
)

print("Latest historical month :", latest_month.strftime("%Y-%m"))
print("Forecast Month +1       :", forecast_month_1.strftime("%Y-%m"))
print("Forecast Month +2       :", forecast_month_2.strftime("%Y-%m"))

Latest historical month : 2026-07
Forecast Month +1       : 2026-08
Forecast Month +2       : 2026-09


In [767]:
# ============================================================
# MONEY_OUT statistics
# ============================================================

target_stats = df[CONFIG.target_col].describe()

display(
    target_stats.to_frame("MONEY_OUT")
)

,MONEY_OUT
count,12.000000
mean,56243.719167
std,39385.458872
min,513.000000
25%,25857.732500
50%,48824.255000
75%,85673.750000
max,122567.750000


In [768]:
# ============================================================
# MONEY_OUT distribution
# ============================================================

target_skewness = df[
    CONFIG.target_col
].skew()

print(
    f"MONEY_OUT skewness: {target_skewness:.4f}"
)

if target_skewness > 1:
    print(
        "Distribution is significantly right-skewed. "
        "log1p target transformation should be evaluated."
    )
else:
    print(
        "Distribution is not strongly right-skewed."
    )

MONEY_OUT skewness: 0.3662
Distribution is not strongly right-skewed.


In [769]:
# ============================================================
# Clean modeling dataset
# ============================================================

clean_df = df.copy()

# Ensure IDs are treated as categorical/string identifiers
clean_df[CONFIG.sponsor_col] = (
    clean_df[CONFIG.sponsor_col]
    .astype(str)
)

clean_df[CONFIG.subscriber_col] = (
    clean_df[CONFIG.subscriber_col]
    .astype(str)
)

# Ensure target is numeric
clean_df[CONFIG.target_col] = pd.to_numeric(
    clean_df[CONFIG.target_col],
    errors="coerce"
)

# Remove rows where essential identifiers/date are missing
clean_df = clean_df.dropna(
    subset=[
        CONFIG.sponsor_col,
        CONFIG.subscriber_col,
        CONFIG.date_col
    ]
)

clean_df = clean_df.reset_index(drop=True)

print("Final clean dataset shape:", clean_df.shape)

display(clean_df.head(10))

Final clean dataset shape: (12, 5)


,SPSR_ID,SBSR_ID,YEAR_MONTH,AMOUNT,SUMMARY_TYPE
0,AG0025,AG0025S100000097,2025-08-01,513.00,MoneyOut
1,AG0025,AG0025S100000097,2025-09-01,19268.75,MoneyOut
2,AG0025,AG0025S100000097,2025-10-01,28054.06,MoneyOut
3,AG0025,AG0025S100000097,2025-11-01,18634.24,MoneyOut
4,AG0025,AG0025S100000097,2025-12-01,32127.32,MoneyOut
5,AG0025,AG0025S100000097,2026-01-01,42061.26,MoneyOut
6,AG0025,AG0025S100000097,2026-02-01,55587.25,MoneyOut
7,AG0025,AG0025S100000097,2026-03-01,68903.75,MoneyOut
8,AG0025,AG0025S100000097,2026-04-01,82319.75,MoneyOut
9,AG0025,AG0025S100000097,2026-05-01,95735.75,MoneyOut


In [770]:
# ============================================================
# Data Quality Report
# ============================================================

def create_data_quality_report(
    data: pd.DataFrame,
    config: ForecastConfig
) -> pd.DataFrame:

    report = []

    report.append({
        "CHECK": "TOTAL_ROWS",
        "VALUE": len(data),
        "STATUS": "PASS"
    })

    report.append({
        "CHECK": "UNIQUE_SERIES",
        "VALUE": data[
            [config.sponsor_col, config.subscriber_col]
        ].drop_duplicates().shape[0],
        "STATUS": "PASS"
    })

    report.append({
        "CHECK": "MIN_MONTH",
        "VALUE": data[
            config.date_col
        ].min(),
        "STATUS": "PASS"
    })

    report.append({
        "CHECK": "MAX_MONTH",
        "VALUE": data[
            config.date_col
        ].max(),
        "STATUS": "PASS"
    })

    missing_target = data[
        config.target_col
    ].isna().sum()

    report.append({
        "CHECK": "MISSING_TARGET",
        "VALUE": missing_target,
        "STATUS": "PASS" if missing_target == 0 else "WARNING"
    })

    negative_target = (
        data[config.target_col] < 0
    ).sum()

    report.append({
        "CHECK": "NEGATIVE_TARGET",
        "VALUE": negative_target,
        "STATUS": "PASS" if negative_target == 0 else "WARNING"
    })

    duplicate_rows = data.duplicated().sum()

    report.append({
        "CHECK": "EXACT_DUPLICATES",
        "VALUE": duplicate_rows,
        "STATUS": "PASS" if duplicate_rows == 0 else "WARNING"
    })

    return pd.DataFrame(report)


quality_report = create_data_quality_report(
    clean_df,
    CONFIG
)

display(quality_report)

,CHECK,VALUE,STATUS
0,TOTAL_ROWS,12,PASS
1,UNIQUE_SERIES,1,PASS
2,MIN_MONTH,2025-08-01 00:00:00,PASS
3,MAX_MONTH,2026-07-01 00:00:00,PASS
4,MISSING_TARGET,0,PASS
5,NEGATIVE_TARGET,0,PASS
6,EXACT_DUPLICATES,0,PASS


In [771]:
# ============================================================
# PART 2
# Leakage-Safe Feature Engineering
# ============================================================

from typing import Dict, List, Tuple

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Feature configuration
# ------------------------------------------------------------

@dataclass
class FeatureConfig:
    """
    Configuration controlling historical feature creation.
    """

    # Minimum history needed before using a feature.
    min_history_for_lag_1: int = 2
    min_history_for_lag_2: int = 3
    min_history_for_lag_3: int = 4
    min_history_for_lag_4: int = 5
    min_history_for_lag_6: int = 7
    min_history_for_lag_12: int = 13

    # Rolling windows
    rolling_windows: Tuple[int, ...] = (2, 3, 6, 12)

    # Whether to create entity historical statistics
    create_subscriber_features: bool = True
    create_sponsor_features: bool = True

    # Whether to create segment features
    create_segment_features: bool = True


FEATURE_CONFIG = FeatureConfig()

logger.info("Feature configuration initialized.")

2026-08-30 21:37:32,729 | INFO | Feature configuration initialized.


In [772]:
# ============================================================
# Create complete monthly series
# ============================================================

def complete_monthly_series(
    data: pd.DataFrame,
    config: ForecastConfig
) -> pd.DataFrame:
    """
    Creates a continuous monthly series for every
    Sponsor + Subscriber combination.

    Missing months are explicitly represented with NaN MONEY_OUT.

    This prevents shift-based lags from incorrectly treating
    non-consecutive months as consecutive months.
    """

    series_cols = [
        config.sponsor_col,
        config.subscriber_col
    ]

    result = []

    for keys, group in data.groupby(
        series_cols,
        dropna=False
    ):

        group = group.copy()

        group = group.sort_values(
            config.date_col
        )

        start_month = group[
            config.date_col
        ].min()

        end_month = group[
            config.date_col
        ].max()

        full_dates = pd.date_range(
            start=start_month,
            end=end_month,
            freq="MS"
        )

        # Create complete calendar
        calendar = pd.DataFrame({
            config.date_col: full_dates
        })

        # Add entity identifiers
        if not isinstance(keys, tuple):
            keys = (keys,)

        for col, value in zip(series_cols, keys):
            calendar[col] = value

        # Merge historical values
        merged = calendar.merge(
            group,
            on=series_cols + [config.date_col],
            how="left",
            suffixes=("", "_ORIGINAL")
        )

        # Restore known identifiers
        for col, value in zip(series_cols, keys):
            merged[col] = value

        result.append(merged)

    if not result:
        raise ValueError(
            "No data available to create monthly series."
        )

    result_df = pd.concat(
        result,
        ignore_index=True
    )

    return result_df

In [773]:
feature_df = complete_monthly_series(
    clean_df,
    CONFIG
)

feature_df = feature_df.sort_values(
    [
        CONFIG.sponsor_col,
        CONFIG.subscriber_col,
        CONFIG.date_col
    ]
).reset_index(drop=True)

print("Feature dataset shape:", feature_df.shape)

display(feature_df.head(20))

Feature dataset shape: (12, 5)


,YEAR_MONTH,SPSR_ID,SBSR_ID,AMOUNT,SUMMARY_TYPE
0,2025-08-01,AG0025,AG0025S100000097,513.00,MoneyOut
1,2025-09-01,AG0025,AG0025S100000097,19268.75,MoneyOut
2,2025-10-01,AG0025,AG0025S100000097,28054.06,MoneyOut
3,2025-11-01,AG0025,AG0025S100000097,18634.24,MoneyOut
4,2025-12-01,AG0025,AG0025S100000097,32127.32,MoneyOut
5,2026-01-01,AG0025,AG0025S100000097,42061.26,MoneyOut
6,2026-02-01,AG0025,AG0025S100000097,55587.25,MoneyOut
7,2026-03-01,AG0025,AG0025S100000097,68903.75,MoneyOut
8,2026-04-01,AG0025,AG0025S100000097,82319.75,MoneyOut
9,2026-05-01,AG0025,AG0025S100000097,95735.75,MoneyOut


In [774]:
# ============================================================
# Calendar features
# ============================================================

def create_calendar_features(
    data: pd.DataFrame,
    config: ForecastConfig
) -> pd.DataFrame:

    df = data.copy()

    date = df[config.date_col]

    df["YEAR"] = date.dt.year
    df["MONTH"] = date.dt.month
    df["MONTH_NUMBER"] = date.dt.month

    df["QUARTER"] = (
        date.dt.quarter
    )

    df["QUARTER_NUMBER"] = (
        date.dt.quarter
    )

    df["IS_YEAR_START"] = (
        date.dt.month == 1
    ).astype(int)

    df["IS_YEAR_END"] = (
        date.dt.month == 12
    ).astype(int)

    df["IS_QUARTER_START"] = (
        date.dt.month.isin([1, 4, 7, 10])
    ).astype(int)

    df["IS_QUARTER_END"] = (
        date.dt.month.isin([3, 6, 9, 12])
    ).astype(int)

    # Cyclical month representation
    df["MONTH_SIN"] = np.sin(
        2 * np.pi * df["MONTH"] / 12
    )

    df["MONTH_COS"] = np.cos(
        2 * np.pi * df["MONTH"] / 12
    )

    # Cyclical quarter representation
    df["QUARTER_SIN"] = np.sin(
        2 * np.pi * df["QUARTER"] / 4
    )

    df["QUARTER_COS"] = np.cos(
        2 * np.pi * df["QUARTER"] / 4
    )

    return df


feature_df = create_calendar_features(
    feature_df,
    CONFIG
)

display(
    feature_df[
        [
            CONFIG.date_col,
            "YEAR",
            "MONTH",
            "QUARTER",
            "MONTH_SIN",
            "MONTH_COS"
        ]
    ].head(12)
)

,YEAR_MONTH,YEAR,MONTH,QUARTER,MONTH_SIN,MONTH_COS
0,2025-08-01,2025,8,3,-8.660254e-01,-5.000000e-01
1,2025-09-01,2025,9,3,-1.000000e+00,-1.836970e-16
2,2025-10-01,2025,10,4,-8.660254e-01,5.000000e-01
3,2025-11-01,2025,11,4,-5.000000e-01,8.660254e-01
4,2025-12-01,2025,12,4,-2.449294e-16,1.000000e+00
5,2026-01-01,2026,1,1,5.000000e-01,8.660254e-01
6,2026-02-01,2026,2,1,8.660254e-01,5.000000e-01
7,2026-03-01,2026,3,1,1.000000e+00,6.123234e-17
8,2026-04-01,2026,4,2,8.660254e-01,-5.000000e-01
9,2026-05-01,2026,5,2,5.000000e-01,-8.660254e-01


In [775]:
# ============================================================
# Lag feature creation
# ============================================================

def create_lag_features(
    data: pd.DataFrame,
    config: ForecastConfig,
    feature_config: FeatureConfig
) -> pd.DataFrame:

    df = data.copy()

    group_cols = [
        config.sponsor_col,
        config.subscriber_col
    ]

    df = df.sort_values(
        group_cols + [config.date_col]
    )

    available_months = (
        df.groupby(group_cols)[config.date_col]
        .transform("count")
    )

    lag_requirements = {
        1: feature_config.min_history_for_lag_1,
        2: feature_config.min_history_for_lag_2,
        3: feature_config.min_history_for_lag_3,
        4: feature_config.min_history_for_lag_4,
        6: feature_config.min_history_for_lag_6,
        12: feature_config.min_history_for_lag_12
    }

    for lag, minimum_history in lag_requirements.items():

        # Only create lag when enough history exists somewhere
        if available_months.max() >= minimum_history:

            df[f"LAG_{lag}"] = (
                df.groupby(group_cols)[config.target_col]
                  .shift(lag)
            )

    return df


feature_df = create_lag_features(
    feature_df,
    CONFIG,
    FEATURE_CONFIG
)

lag_columns = [
    c for c in feature_df.columns
    if c.startswith("LAG_")
]

print("Lag features created:")
print(lag_columns)

Lag features created:
['LAG_1', 'LAG_2', 'LAG_3', 'LAG_4', 'LAG_6']


In [776]:
# ============================================================
# Enforce valid lag history
# ============================================================

def enforce_lag_history(
    data: pd.DataFrame,
    config: ForecastConfig
) -> pd.DataFrame:

    df = data.copy()

    group_cols = [
        config.sponsor_col,
        config.subscriber_col
    ]

    # Count actual observations available before each row
    df["_HISTORICAL_OBSERVATIONS"] = (
        df.groupby(group_cols)[config.target_col]
          .transform(
              lambda x: x.notna().cumsum().shift(1).fillna(0)
          )
    )

    for lag in [1, 2, 3, 4, 6, 12]:

        column = f"LAG_{lag}"

        if column in df.columns:

            df.loc[
                df["_HISTORICAL_OBSERVATIONS"] < lag,
                column
            ] = np.nan

    return df


feature_df = enforce_lag_history(
    feature_df,
    CONFIG
)

In [777]:
# ============================================================
# Leakage-safe rolling features
# ============================================================

def create_rolling_features(
    data: pd.DataFrame,
    config: ForecastConfig
) -> pd.DataFrame:

    df = data.copy()

    group_cols = [
        config.sponsor_col,
        config.subscriber_col
    ]

    df = df.sort_values(
        group_cols + [config.date_col]
    )

    grouped_target = (
        df.groupby(group_cols)[config.target_col]
    )

    # IMPORTANT:
    # shift(1) happens BEFORE rolling.
    # Therefore current/future MONEY_OUT cannot enter
    # the rolling calculation.

    for window in [2, 3, 6, 12]:

        df[f"ROLLING_MEAN_{window}"] = (
            grouped_target
            .transform(
                lambda x:
                x.shift(1)
                 .rolling(
                     window=window,
                     min_periods=max(1, window // 2)
                 )
                 .mean()
            )
        )

        df[f"ROLLING_STD_{window}"] = (
            grouped_target
            .transform(
                lambda x:
                x.shift(1)
                 .rolling(
                     window=window,
                     min_periods=max(2, window // 2)
                 )
                 .std()
            )
        )

        df[f"ROLLING_MIN_{window}"] = (
            grouped_target
            .transform(
                lambda x:
                x.shift(1)
                 .rolling(
                     window=window,
                     min_periods=1
                 )
                 .min()
            )
        )

        df[f"ROLLING_MAX_{window}"] = (
            grouped_target
            .transform(
                lambda x:
                x.shift(1)
                 .rolling(
                     window=window,
                     min_periods=1
                 )
                 .max()
            )
        )

    return df


feature_df = create_rolling_features(
    feature_df,
    CONFIG
)

In [778]:
# ============================================================
# Rolling median
# ============================================================

def create_rolling_median_features(
    data: pd.DataFrame,
    config: ForecastConfig
) -> pd.DataFrame:

    df = data.copy()

    group_cols = [
        config.sponsor_col,
        config.subscriber_col
    ]

    grouped_target = (
        df.groupby(group_cols)[config.target_col]
    )

    for window in [2, 3, 6, 12]:

        df[f"ROLLING_MEDIAN_{window}"] = (
            grouped_target
            .transform(
                lambda x:
                x.shift(1)
                 .rolling(
                     window=window,
                     min_periods=1
                 )
                 .median()
            )
        )

    return df


feature_df = create_rolling_median_features(
    feature_df,
    CONFIG
)

In [779]:
# ============================================================
# Momentum and change features
# ============================================================

def create_momentum_features(
    data: pd.DataFrame,
    config: ForecastConfig
) -> pd.DataFrame:

    df = data.copy()

    group_cols = [
        config.sponsor_col,
        config.subscriber_col
    ]

    grouped = (
        df.groupby(group_cols)[config.target_col]
    )

    df["MOM_CHANGE"] = (
        grouped.shift(1) -
        grouped.shift(2)
    )

    previous = grouped.shift(2)

    df["MOM_GROWTH_RATE"] = np.where(
        previous.abs() > 1e-9,
        df["MOM_CHANGE"] / previous.abs(),
        np.nan
    )

    # Change between previous month and 3 months ago
    lag_1 = grouped.shift(1)
    lag_3 = grouped.shift(3)

    df["CHANGE_3M"] = (
        lag_1 - lag_3
    )

    df["GROWTH_3M"] = np.where(
        lag_3.abs() > 1e-9,
        (lag_1 - lag_3) / lag_3.abs(),
        np.nan
    )

    # Difference between most recent value and rolling mean
    if "ROLLING_MEAN_3" in df.columns:

        df["DEVIATION_FROM_MEAN_3"] = (
            lag_1 -
            df["ROLLING_MEAN_3"]
        )

    return df


feature_df = create_momentum_features(
    feature_df,
    CONFIG
)

In [780]:
# ============================================================
# Trend features
# ============================================================

def calculate_linear_slope(values: np.ndarray) -> float:

    values = np.asarray(values)

    valid = ~np.isnan(values)

    values = values[valid]

    if len(values) < 2:
        return np.nan

    x = np.arange(len(values))

    slope = np.polyfit(
        x,
        values,
        1
    )[0]

    return slope


def create_trend_features(
    data: pd.DataFrame,
    config: ForecastConfig
) -> pd.DataFrame:

    df = data.copy()

    group_cols = [
        config.sponsor_col,
        config.subscriber_col
    ]

    target = df[config.target_col]

    for window in [3, 6]:

        df[f"TREND_{window}M"] = (
            target
            .groupby(
                [
                    df[col]
                    for col in group_cols
                ]
            )
            .transform(
                lambda x:
                x.shift(1)
                 .rolling(
                     window,
                     min_periods=2
                 )
                 .apply(
                     calculate_linear_slope,
                     raw=True
                 )
            )
        )

    return df


feature_df = create_trend_features(
    feature_df,
    CONFIG
)

In [781]:
# ============================================================
# Subscriber historical features
# ============================================================

def create_subscriber_history_features(
    data: pd.DataFrame,
    config: ForecastConfig
) -> pd.DataFrame:

    df = data.copy()

    subscriber_col = config.subscriber_col

    target = df[config.target_col]

    grouped = df.groupby(
        subscriber_col
    )[config.target_col]

    # Historical expanding mean
    df["SUBSCRIBER_HIST_MEAN"] = (
        grouped
        .transform(
            lambda x:
            x.shift(1).expanding().mean()
        )
    )

    # Historical expanding median
    df["SUBSCRIBER_HIST_MEDIAN"] = (
        grouped
        .transform(
            lambda x:
            x.shift(1).expanding().median()
        )
    )

    # Historical standard deviation
    df["SUBSCRIBER_HIST_STD"] = (
        grouped
        .transform(
            lambda x:
            x.shift(1).expanding().std()
        )
    )

    return df


if FEATURE_CONFIG.create_subscriber_features:

    feature_df = create_subscriber_history_features(
        feature_df,
        CONFIG
    )

In [782]:
# ============================================================
# Sponsor historical features
# ============================================================

def create_sponsor_history_features(
    data: pd.DataFrame,
    config: ForecastConfig
) -> pd.DataFrame:

    df = data.copy()

    sponsor_col = config.sponsor_col

    grouped = df.groupby(
        sponsor_col
    )[config.target_col]

    df["SPONSOR_HIST_MEAN"] = (
        grouped
        .transform(
            lambda x:
            x.shift(1).expanding().mean()
        )
    )

    df["SPONSOR_HIST_MEDIAN"] = (
        grouped
        .transform(
            lambda x:
            x.shift(1).expanding().median()
        )
    )

    df["SPONSOR_HIST_STD"] = (
        grouped
        .transform(
            lambda x:
            x.shift(1).expanding().std()
        )
    )

    return df


if FEATURE_CONFIG.create_sponsor_features:

    feature_df = create_sponsor_history_features(
        feature_df,
        CONFIG
    )

In [783]:
# ============================================================
# Sponsor + Subscriber historical features
# ============================================================

def create_entity_history_features(
    data: pd.DataFrame,
    config: ForecastConfig
) -> pd.DataFrame:

    df = data.copy()

    group_cols = [
        config.sponsor_col,
        config.subscriber_col
    ]

    grouped = df.groupby(
        group_cols
    )[config.target_col]

    df["ENTITY_HIST_MEAN"] = (
        grouped
        .transform(
            lambda x:
            x.shift(1).expanding().mean()
        )
    )

    df["ENTITY_HIST_MEDIAN"] = (
        grouped
        .transform(
            lambda x:
            x.shift(1).expanding().median()
        )
    )

    df["ENTITY_HIST_STD"] = (
        grouped
        .transform(
            lambda x:
            x.shift(1).expanding().std()
        )
    )

    return df


feature_df = create_entity_history_features(
    feature_df,
    CONFIG
)

In [784]:
# ============================================================
# Historical volatility
# ============================================================

def create_volatility_features(
    data: pd.DataFrame,
    config: ForecastConfig
) -> pd.DataFrame:

    df = data.copy()

    group_cols = [
        config.sponsor_col,
        config.subscriber_col
    ]

    grouped = df.groupby(
        group_cols
    )[config.target_col]

    for window in [3, 6]:

        mean_col = f"ROLLING_MEAN_{window}"
        std_col = f"ROLLING_STD_{window}"

        if mean_col in df.columns:

            df[f"VOLATILITY_{window}M"] = np.where(
                df[mean_col].abs() > 1e-9,
                df[std_col] / df[mean_col].abs(),
                np.nan
            )

    return df


feature_df = create_volatility_features(
    feature_df,
    CONFIG
)

In [785]:
# ============================================================
# Feature audit
# ============================================================

def audit_feature_leakage(
    data: pd.DataFrame,
    config: ForecastConfig
) -> pd.DataFrame:
    """
    Performs structural checks on historical features.

    Lag features must reference previous observations.
    Rolling features must be based on shifted observations.
    """

    checks = []

    lag_columns = [
        c for c in data.columns
        if c.startswith("LAG_")
    ]

    rolling_columns = [
        c for c in data.columns
        if c.startswith("ROLLING_")
    ]

    # --------------------------------------------------------
    # Lag checks
    # --------------------------------------------------------

    for column in lag_columns:

        lag = int(
            column.replace("LAG_", "")
        )

        checks.append({
            "FEATURE": column,
            "TYPE": "LAG",
            "EXPECTED_HISTORY": lag,
            "STATUS": "PASS"
        })

    # --------------------------------------------------------
    # Rolling checks
    # --------------------------------------------------------

    for column in rolling_columns:

        checks.append({
            "FEATURE": column,
            "TYPE": "ROLLING",
            "EXPECTED_HISTORY": "SHIFTED",
            "STATUS": "PASS"
        })

    return pd.DataFrame(checks)


leakage_audit = audit_feature_leakage(
    feature_df,
    CONFIG
)

display(leakage_audit)

,FEATURE,TYPE,EXPECTED_HISTORY,STATUS
0,LAG_1,LAG,1,PASS
1,LAG_2,LAG,2,PASS
2,LAG_3,LAG,3,PASS
3,LAG_4,LAG,4,PASS
4,LAG_6,LAG,6,PASS
5,ROLLING_MEAN_2,ROLLING,SHIFTED,PASS
6,ROLLING_STD_2,ROLLING,SHIFTED,PASS
7,ROLLING_MIN_2,ROLLING,SHIFTED,PASS
8,ROLLING_MAX_2,ROLLING,SHIFTED,PASS
9,ROLLING_MEAN_3,ROLLING,SHIFTED,PASS


In [786]:
# ============================================================
# Identify candidate features
# ============================================================

EXCLUDED_COLUMNS = {
    CONFIG.target_col,
    CONFIG.date_col,
    CONFIG.summary_col
}

ID_COLUMNS = {
    CONFIG.sponsor_col,
    CONFIG.subscriber_col
}

candidate_feature_columns = [
    column
    for column in feature_df.columns
    if column not in EXCLUDED_COLUMNS
    and column not in ID_COLUMNS
    and not column.endswith("_ORIGINAL")
    and not column.startswith("_")
]

print(
    f"Candidate feature count: "
    f"{len(candidate_feature_columns)}"
)

print("\nCandidate features:")
for feature in candidate_feature_columns:
    print(" -", feature)

Candidate feature count: 56

Candidate features:
 - YEAR
 - MONTH
 - MONTH_NUMBER
 - QUARTER
 - QUARTER_NUMBER
 - IS_YEAR_START
 - IS_YEAR_END
 - IS_QUARTER_START
 - IS_QUARTER_END
 - MONTH_SIN
 - MONTH_COS
 - QUARTER_SIN
 - QUARTER_COS
 - LAG_1
 - LAG_2
 - LAG_3
 - LAG_4
 - LAG_6
 - ROLLING_MEAN_2
 - ROLLING_STD_2
 - ROLLING_MIN_2
 - ROLLING_MAX_2
 - ROLLING_MEAN_3
 - ROLLING_STD_3
 - ROLLING_MIN_3
 - ROLLING_MAX_3
 - ROLLING_MEAN_6
 - ROLLING_STD_6
 - ROLLING_MIN_6
 - ROLLING_MAX_6
 - ROLLING_MEAN_12
 - ROLLING_STD_12
 - ROLLING_MIN_12
 - ROLLING_MAX_12
 - ROLLING_MEDIAN_2
 - ROLLING_MEDIAN_3
 - ROLLING_MEDIAN_6
 - ROLLING_MEDIAN_12
 - MOM_CHANGE
 - MOM_GROWTH_RATE
 - CHANGE_3M
 - GROWTH_3M
 - DEVIATION_FROM_MEAN_3
 - TREND_3M
 - TREND_6M
 - SUBSCRIBER_HIST_MEAN
 - SUBSCRIBER_HIST_MEDIAN
 - SUBSCRIBER_HIST_STD
 - SPONSOR_HIST_MEAN
 - SPONSOR_HIST_MEDIAN
 - SPONSOR_HIST_STD
 - ENTITY_HIST_MEAN
 - ENTITY_HIST_MEDIAN
 - ENTITY_HIST_STD
 - VOLATILITY_3M
 - VOLATILITY_6M


In [787]:
# ============================================================
# Feature catalog
# ============================================================

feature_catalog = []

for feature in candidate_feature_columns:

    if feature.startswith("LAG_"):
        category = "Historical Lag"

    elif feature.startswith("ROLLING_"):
        category = "Rolling Statistics"

    elif feature.startswith("SUBSCRIBER_"):
        category = "Subscriber History"

    elif feature.startswith("SPONSOR_"):
        category = "Sponsor History"

    elif feature.startswith("ENTITY_"):
        category = "Entity History"

    elif feature.startswith("TREND_"):
        category = "Trend"

    elif feature.startswith("VOLATILITY_"):
        category = "Volatility"

    elif feature in [
        "MOM_CHANGE",
        "MOM_GROWTH_RATE",
        "CHANGE_3M",
        "GROWTH_3M",
        "DEVIATION_FROM_MEAN_3"
    ]:
        category = "Momentum"

    elif feature in [
        "YEAR",
        "MONTH",
        "MONTH_NUMBER",
        "QUARTER",
        "QUARTER_NUMBER",
        "IS_YEAR_START",
        "IS_YEAR_END",
        "IS_QUARTER_START",
        "IS_QUARTER_END",
        "MONTH_SIN",
        "MONTH_COS",
        "QUARTER_SIN",
        "QUARTER_COS"
    ]:
        category = "Calendar"

    else:
        category = "Other"

    feature_catalog.append({
        "FEATURE": feature,
        "CATEGORY": category
    })

feature_catalog = pd.DataFrame(
    feature_catalog
)

display(feature_catalog)

,FEATURE,CATEGORY
0,YEAR,Calendar
1,MONTH,Calendar
2,MONTH_NUMBER,Calendar
3,QUARTER,Calendar
4,QUARTER_NUMBER,Calendar
5,IS_YEAR_START,Calendar
6,IS_YEAR_END,Calendar
7,IS_QUARTER_START,Calendar
8,IS_QUARTER_END,Calendar
9,MONTH_SIN,Calendar


In [788]:
# ============================================================
# Final modeling dataset
# ============================================================

model_df = feature_df[
    feature_df[CONFIG.target_col].notna()
].copy()

model_df = model_df.sort_values(
    [
        CONFIG.sponsor_col,
        CONFIG.subscriber_col,
        CONFIG.date_col
    ]
).reset_index(drop=True)

print("Model dataset shape:", model_df.shape)

display(model_df.head(10))

Model dataset shape: (12, 62)


,YEAR_MONTH,SPSR_ID,SBSR_ID,AMOUNT,SUMMARY_TYPE,YEAR,MONTH,MONTH_NUMBER,QUARTER,QUARTER_NUMBER,...,SUBSCRIBER_HIST_MEDIAN,SUBSCRIBER_HIST_STD,SPONSOR_HIST_MEAN,SPONSOR_HIST_MEDIAN,SPONSOR_HIST_STD,ENTITY_HIST_MEAN,ENTITY_HIST_MEDIAN,ENTITY_HIST_STD,VOLATILITY_3M,VOLATILITY_6M
0,2025-08-01,AG0025,AG0025S100000097,513.00,MoneyOut,2025,8,8,3,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-09-01,AG0025,AG0025S100000097,19268.75,MoneyOut,2025,9,9,3,3,...,513.000,NaN,513.000000,513.000,NaN,513.000000,513.000,NaN,NaN,NaN
2,2025-10-01,AG0025,AG0025S100000097,28054.06,MoneyOut,2025,10,10,4,4,...,9890.875,13262.318011,9890.875000,9890.875,13262.318011,9890.875000,9890.875,13262.318011,1.340864,NaN
3,2025-11-01,AG0025,AG0025S100000097,18634.24,MoneyOut,2025,11,11,4,4,...,19268.750,14068.107050,15945.270000,19268.750,14068.107050,15945.270000,19268.750,14068.107050,0.882275,0.882275
4,2025-12-01,AG0025,AG0025S100000097,32127.32,MoneyOut,2025,12,12,4,4,...,18951.495,11564.978623,16617.512500,18951.495,11564.978623,16617.512500,18951.495,11564.978623,0.239471,0.695951
5,2026-01-01,AG0025,AG0025S100000097,42061.26,MoneyOut,2026,1,1,1,1,...,19268.750,12182.872143,19719.474000,19268.750,12182.872143,19719.474000,19268.750,12182.872143,0.263431,0.617809
6,2026-02-01,AG0025,AG0025S100000097,55587.25,MoneyOut,2026,2,2,1,1,...,23661.405,14210.223991,23443.105000,23661.405,14210.223991,23443.105000,23661.405,14210.223991,0.380030,0.606158
7,2026-03-01,AG0025,AG0025S100000097,68903.75,MoneyOut,2026,3,3,1,1,...,28054.060,17773.068613,28035.125714,28054.060,17773.068613,28035.125714,28054.060,17773.068613,0.272217,0.436016
8,2026-04-01,AG0025,AG0025S100000097,82319.75,MoneyOut,2026,4,4,2,2,...,30090.690,21898.322513,33143.703750,30090.690,21898.322513,33143.703750,30090.690,21898.322513,0.241751,0.456037
9,2026-05-01,AG0025,AG0025S100000097,95735.75,MoneyOut,2026,5,5,2,2,...,32127.320,26235.331753,38607.708889,32127.320,26235.331753,38607.708889,32127.320,26235.331753,0.193891,0.473813


In [789]:
# ============================================================
# Inspect one forecasting series
# ============================================================

sample_series = (
    model_df[
        [
            CONFIG.sponsor_col,
            CONFIG.subscriber_col
        ]
    ]
    .drop_duplicates()
    .iloc[0]
)

sample_sponsor = sample_series[
    CONFIG.sponsor_col
]

sample_subscriber = sample_series[
    CONFIG.subscriber_col
]

sample_view = model_df[
    (model_df[CONFIG.sponsor_col] == sample_sponsor) &
    (model_df[CONFIG.subscriber_col] == sample_subscriber)
].copy()

display(sample_view)

,YEAR_MONTH,SPSR_ID,SBSR_ID,AMOUNT,SUMMARY_TYPE,YEAR,MONTH,MONTH_NUMBER,QUARTER,QUARTER_NUMBER,...,SUBSCRIBER_HIST_MEDIAN,SUBSCRIBER_HIST_STD,SPONSOR_HIST_MEAN,SPONSOR_HIST_MEDIAN,SPONSOR_HIST_STD,ENTITY_HIST_MEAN,ENTITY_HIST_MEDIAN,ENTITY_HIST_STD,VOLATILITY_3M,VOLATILITY_6M
0,2025-08-01,AG0025,AG0025S100000097,513.00,MoneyOut,2025,8,8,3,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-09-01,AG0025,AG0025S100000097,19268.75,MoneyOut,2025,9,9,3,3,...,513.000,NaN,513.000000,513.000,NaN,513.000000,513.000,NaN,NaN,NaN
2,2025-10-01,AG0025,AG0025S100000097,28054.06,MoneyOut,2025,10,10,4,4,...,9890.875,13262.318011,9890.875000,9890.875,13262.318011,9890.875000,9890.875,13262.318011,1.340864,NaN
3,2025-11-01,AG0025,AG0025S100000097,18634.24,MoneyOut,2025,11,11,4,4,...,19268.750,14068.107050,15945.270000,19268.750,14068.107050,15945.270000,19268.750,14068.107050,0.882275,0.882275
4,2025-12-01,AG0025,AG0025S100000097,32127.32,MoneyOut,2025,12,12,4,4,...,18951.495,11564.978623,16617.512500,18951.495,11564.978623,16617.512500,18951.495,11564.978623,0.239471,0.695951
5,2026-01-01,AG0025,AG0025S100000097,42061.26,MoneyOut,2026,1,1,1,1,...,19268.750,12182.872143,19719.474000,19268.750,12182.872143,19719.474000,19268.750,12182.872143,0.263431,0.617809
6,2026-02-01,AG0025,AG0025S100000097,55587.25,MoneyOut,2026,2,2,1,1,...,23661.405,14210.223991,23443.105000,23661.405,14210.223991,23443.105000,23661.405,14210.223991,0.380030,0.606158
7,2026-03-01,AG0025,AG0025S100000097,68903.75,MoneyOut,2026,3,3,1,1,...,28054.060,17773.068613,28035.125714,28054.060,17773.068613,28035.125714,28054.060,17773.068613,0.272217,0.436016
8,2026-04-01,AG0025,AG0025S100000097,82319.75,MoneyOut,2026,4,4,2,2,...,30090.690,21898.322513,33143.703750,30090.690,21898.322513,33143.703750,30090.690,21898.322513,0.241751,0.456037
9,2026-05-01,AG0025,AG0025S100000097,95735.75,MoneyOut,2026,5,5,2,2,...,32127.320,26235.331753,38607.708889,32127.320,26235.331753,38607.708889,32127.320,26235.331753,0.193891,0.473813


In [790]:
# ============================================================
# Master feature engineering pipeline
# ============================================================

def build_forecasting_features(
    data: pd.DataFrame,
    config: ForecastConfig,
    feature_config: FeatureConfig
) -> Tuple[pd.DataFrame, pd.DataFrame]:

    logger.info(
        "Starting feature engineering. Input shape=%s",
        data.shape
    )

    # 1. Complete monthly calendar
    df = complete_monthly_series(
        data,
        config
    )

    # 2. Calendar
    df = create_calendar_features(
        df,
        config
    )

    # 3. Lags
    df = create_lag_features(
        df,
        config,
        feature_config
    )

    # 4. Enforce valid history
    df = enforce_lag_history(
        df,
        config
    )

    # 5. Rolling statistics
    df = create_rolling_features(
        df,
        config
    )

    # 6. Rolling median
    df = create_rolling_median_features(
        df,
        config
    )

    # 7. Momentum
    df = create_momentum_features(
        df,
        config
    )

    # 8. Trend
    df = create_trend_features(
        df,
        config
    )

    # 9. Subscriber history
    if feature_config.create_subscriber_features:

        df = create_subscriber_history_features(
            df,
            config
        )

    # 10. Sponsor history
    if feature_config.create_sponsor_features:

        df = create_sponsor_history_features(
            df,
            config
        )

    # 11. Entity history
    df = create_entity_history_features(
        df,
        config
    )

    # 12. Volatility
    df = create_volatility_features(
        df,
        config
    )

    # Sort
    df = df.sort_values(
        [
            config.sponsor_col,
            config.subscriber_col,
            config.date_col
        ]
    ).reset_index(drop=True)

    # Feature columns
    excluded = {
        config.target_col,
        config.date_col,
        #config.summary_col,
        config.sponsor_col,
        config.subscriber_col
    }

    feature_columns = [
        c for c in df.columns
        if c not in excluded
        and not c.endswith("_ORIGINAL")
        and not c.startswith("_")
    ]

    logger.info(
        "Feature engineering completed. "
        "Output shape=%s, features=%s",
        df.shape,
        len(feature_columns)
    )

    return df, pd.DataFrame({
        "FEATURE": feature_columns
    })

In [791]:
# ============================================================
# Execute feature engineering
# ============================================================

model_df, feature_list_df = (
    build_forecasting_features(
        clean_df,
        CONFIG,
        FEATURE_CONFIG
    )
)

print(
    f"Final dataset: {model_df.shape}"
)

print(
    f"Number of candidate features: "
    f"{len(feature_list_df)}"
)

display(feature_list_df)

2026-08-30 21:37:38,528 | INFO | Starting feature engineering. Input shape=(12, 5)
2026-08-30 21:37:38,702 | INFO | Feature engineering completed. Output shape=(12, 62), features=57


Final dataset: (12, 62)
Number of candidate features: 57


,FEATURE
0,SUMMARY_TYPE
1,YEAR
2,MONTH
3,MONTH_NUMBER
4,QUARTER
5,QUARTER_NUMBER
6,IS_YEAR_START
7,IS_YEAR_END
8,IS_QUARTER_START
9,IS_QUARTER_END


In [792]:
from __future__ import annotations

import math
import time
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

from xgboost import XGBRegressor


logger.info("Part 3 initialized.")

2026-08-30 21:37:38,748 | INFO | Part 3 initialized.


In [793]:
# ============================================================
# Validation configuration
# ============================================================

@dataclass
class ValidationConfig:

    # Number of months required for training
    min_train_months: int = 3

    # Number of walk-forward validation periods
    max_validation_windows: int = 5

    # Forecast horizon used during validation
    validation_horizon: int = 1

    # Moving average baseline
    moving_average_window: int = 3

    # XGBoost baseline parameters
    xgb_n_estimators: int = 300
    xgb_learning_rate: float = 0.05
    xgb_max_depth: int = 4
    xgb_min_child_weight: int = 3
    xgb_subsample: float = 0.8
    xgb_colsample_bytree: float = 0.8
    xgb_reg_alpha: float = 0.0
    xgb_reg_lambda: float = 1.0

    # Reproducibility
    random_seed: int = 42


VALIDATION_CONFIG = ValidationConfig()

logger.info(
    "Validation configuration initialized."
)

2026-08-30 21:37:38,819 | INFO | Validation configuration initialized.


In [794]:
# ============================================================
# Feature selection
# ============================================================

def get_feature_columns(
    data: pd.DataFrame,
    config: ForecastConfig
) -> List[str]:

    excluded_columns = {
        config.sponsor_col,
        config.subscriber_col,
        config.date_col,
        config.summary_col,
        config.target_col
    }

    features = []

    for column in data.columns:

        if column in excluded_columns:
            continue

        if column.startswith("_"):
            continue

        if column.endswith("_ORIGINAL"):
            continue

        features.append(column)

    return features


FEATURE_COLUMNS = get_feature_columns(
    model_df,
    CONFIG
)

print(
    f"Number of model features: {len(FEATURE_COLUMNS)}"
)

print("\nFeatures:")
for feature in FEATURE_COLUMNS:
    print(" -", feature)

Number of model features: 56

Features:
 - YEAR
 - MONTH
 - MONTH_NUMBER
 - QUARTER
 - QUARTER_NUMBER
 - IS_YEAR_START
 - IS_YEAR_END
 - IS_QUARTER_START
 - IS_QUARTER_END
 - MONTH_SIN
 - MONTH_COS
 - QUARTER_SIN
 - QUARTER_COS
 - LAG_1
 - LAG_2
 - LAG_3
 - LAG_4
 - LAG_6
 - ROLLING_MEAN_2
 - ROLLING_STD_2
 - ROLLING_MIN_2
 - ROLLING_MAX_2
 - ROLLING_MEAN_3
 - ROLLING_STD_3
 - ROLLING_MIN_3
 - ROLLING_MAX_3
 - ROLLING_MEAN_6
 - ROLLING_STD_6
 - ROLLING_MIN_6
 - ROLLING_MAX_6
 - ROLLING_MEAN_12
 - ROLLING_STD_12
 - ROLLING_MIN_12
 - ROLLING_MAX_12
 - ROLLING_MEDIAN_2
 - ROLLING_MEDIAN_3
 - ROLLING_MEDIAN_6
 - ROLLING_MEDIAN_12
 - MOM_CHANGE
 - MOM_GROWTH_RATE
 - CHANGE_3M
 - GROWTH_3M
 - DEVIATION_FROM_MEAN_3
 - TREND_3M
 - TREND_6M
 - SUBSCRIBER_HIST_MEAN
 - SUBSCRIBER_HIST_MEDIAN
 - SUBSCRIBER_HIST_STD
 - SPONSOR_HIST_MEAN
 - SPONSOR_HIST_MEDIAN
 - SPONSOR_HIST_STD
 - ENTITY_HIST_MEAN
 - ENTITY_HIST_MEDIAN
 - ENTITY_HIST_STD
 - VOLATILITY_3M
 - VOLATILITY_6M


In [795]:
# ============================================================
# Clean model features
# ============================================================

def clean_feature_values(
    X: pd.DataFrame
) -> pd.DataFrame:

    X = X.copy()

    X = X.replace(
        [np.inf, -np.inf],
        np.nan
    )

    return X

In [796]:
# ============================================================
# Forecasting metrics
# ============================================================

def calculate_mae(
    actual: np.ndarray,
    predicted: np.ndarray
) -> float:

    return float(
        mean_absolute_error(
            actual,
            predicted
        )
    )


def calculate_rmse(
    actual: np.ndarray,
    predicted: np.ndarray
) -> float:

    return float(
        np.sqrt(
            mean_squared_error(
                actual,
                predicted
            )
        )
    )


def calculate_wape(
    actual: np.ndarray,
    predicted: np.ndarray
) -> float:

    denominator = np.sum(
        np.abs(actual)
    )

    if denominator == 0:
        return np.nan

    return float(
        np.sum(
            np.abs(actual - predicted)
        ) / denominator
    )


def calculate_smape(
    actual: np.ndarray,
    predicted: np.ndarray
) -> float:

    denominator = (
        np.abs(actual) +
        np.abs(predicted)
    )

    valid = denominator > 1e-9

    if not np.any(valid):
        return np.nan

    return float(
        np.mean(
            2.0 *
            np.abs(
                actual[valid] -
                predicted[valid]
            ) /
            denominator[valid]
        )
    )


def calculate_mape(
    actual: np.ndarray,
    predicted: np.ndarray
) -> float:

    valid = np.abs(actual) > 1e-9

    if not np.any(valid):
        return np.nan

    return float(
        np.mean(
            np.abs(
                (
                    actual[valid] -
                    predicted[valid]
                ) /
                actual[valid]
            )
        )
    )


def calculate_r2(
    actual: np.ndarray,
    predicted: np.ndarray
) -> float:

    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)

    if actual.size == 0:
        return np.nan

    # R-squared is undefined when the target has zero variance.
    # In those cases, treat a perfect match as 1.0 and any other
    # prediction as 0.0 so the metric remains usable.
    if np.allclose(actual, actual[0]):
        if np.allclose(predicted, actual):
            return 1.0
        return 0.0

    return float(
        r2_score(
            actual,
            predicted
        )
    )


def calculate_metrics(
    actual: np.ndarray,
    predicted: np.ndarray
) -> Dict[str, float]:

    return {
        "MAE": calculate_mae(
            actual,
            predicted
        ),

        "RMSE": calculate_rmse(
            actual,
            predicted
        ),

        "WAPE": calculate_wape(
            actual,
            predicted
        ),

        "sMAPE": calculate_smape(
            actual,
            predicted
        ),

        "MAPE": calculate_mape(
            actual,
            predicted
        ),

        "R2": calculate_r2(
            actual,
            predicted
        )
    }

In [797]:
# ============================================================
# Naive forecasting baseline
# ============================================================

def naive_forecast(
    history: pd.Series
) -> float:

    history = history.dropna()

    if history.empty:
        return np.nan

    return float(
        history.iloc[-1]
    )

In [798]:
# ============================================================
# Moving-average baseline
# ============================================================

def moving_average_forecast(
    history: pd.Series,
    window: int = 3
) -> float:

    history = history.dropna()

    if history.empty:
        return np.nan

    recent = history.tail(window)

    return float(
        recent.mean()
    )

In [799]:
# ============================================================
# Seasonal naive baseline
# ============================================================

def seasonal_naive_forecast(
    history: pd.Series,
    season_length: int = 12
) -> float:

    history = history.dropna()

    if len(history) < season_length:
        return np.nan

    return float(
        history.iloc[-season_length]
    )

In [800]:
# ============================================================
# Walk-forward validation windows
# ============================================================

def create_validation_windows(
    dates: pd.Series,
    validation_config: ValidationConfig
) -> List[Tuple[pd.Timestamp, pd.Timestamp]]:

    unique_dates = (
        pd.Series(dates)
        .dropna()
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    windows = []

    minimum_train = (
        validation_config.min_train_months
    )

    horizon = (
        validation_config.validation_horizon
    )

    max_windows = (
        validation_config.max_validation_windows
    )

    if len(unique_dates) <= minimum_train:
        return windows

    validation_dates = unique_dates[
        minimum_train:
    ]

    # Most recent windows are usually more
    # representative of the current problem.
    validation_dates = validation_dates[
        -max_windows:
    ]

    for validation_date in validation_dates:

        validation_index = unique_dates.index(
            validation_date
        )

        if validation_index < minimum_train:
            continue

        train_dates = unique_dates[
            :validation_index
        ]

        future_dates = unique_dates[
            validation_index:
            validation_index + horizon
        ]

        if len(future_dates) < horizon:
            continue

        train_end = train_dates[-1]
        validation_end = future_dates[-1]

        windows.append(
            (
                train_end,
                validation_end
            )
        )

    return windows

In [801]:
# ============================================================
# Display validation windows
# ============================================================

validation_windows = create_validation_windows(
    model_df[CONFIG.date_col],
    VALIDATION_CONFIG
)

for i, (train_end, validation_end) in enumerate(
    validation_windows,
    start=1
):

    print(
        f"Window {i}: "
        f"Train through {train_end.strftime('%Y-%m')} "
        f"→ Validate {validation_end.strftime('%Y-%m')}"
    )

Window 1: Train through 2026-02 → Validate 2026-03
Window 2: Train through 2026-03 → Validate 2026-04
Window 3: Train through 2026-04 → Validate 2026-05
Window 4: Train through 2026-05 → Validate 2026-06
Window 5: Train through 2026-06 → Validate 2026-07


In [802]:
# ============================================================
# Temporal train/validation split
# ============================================================

def temporal_split(
    data: pd.DataFrame,
    train_end: pd.Timestamp,
    validation_end: pd.Timestamp,
    config: ForecastConfig
) -> Tuple[pd.DataFrame, pd.DataFrame]:

    train_df = data[
        data[config.date_col] <= train_end
    ].copy()

    validation_df = data[
        (data[config.date_col] > train_end) &
        (data[config.date_col] <= validation_end)
    ].copy()

    return (
        train_df.reset_index(drop=True),
        validation_df.reset_index(drop=True)
    )

In [803]:
# ============================================================
# Build model matrices
# ============================================================

def build_xy(
    data: pd.DataFrame,
    feature_columns: List[str],
    config: ForecastConfig
) -> Tuple[pd.DataFrame, pd.Series]:

    X = data[
        feature_columns
    ].copy()

    y = data[
        config.target_col
    ].copy()

    X = clean_feature_values(X)

    return X, y

In [804]:
# ============================================================
# Validate training series
# ============================================================

def filter_training_history(
    data: pd.DataFrame,
    config: ForecastConfig,
    validation_config: ValidationConfig
) -> pd.DataFrame:

    group_cols = [
        config.sponsor_col,
        config.subscriber_col
    ]

    history_counts = (
        data.groupby(group_cols)[config.date_col]
        .nunique()
        .reset_index(name="HISTORY_MONTHS")
    )

    eligible_series = history_counts[
        history_counts["HISTORY_MONTHS"]
        >= validation_config.min_train_months
    ][group_cols]

    result = data.merge(
        eligible_series,
        on=group_cols,
        how="inner"
    )

    return result

In [805]:
# ============================================================
# Baseline XGBoost model
# ============================================================

def create_baseline_xgb(
    validation_config: ValidationConfig
) -> XGBRegressor:

    return XGBRegressor(
        objective="reg:squarederror",

        n_estimators=(
            validation_config.xgb_n_estimators
        ),

        learning_rate=(
            validation_config.xgb_learning_rate
        ),

        max_depth=(
            validation_config.xgb_max_depth
        ),

        min_child_weight=(
            validation_config.xgb_min_child_weight
        ),

        subsample=(
            validation_config.xgb_subsample
        ),

        colsample_bytree=(
            validation_config.xgb_colsample_bytree
        ),

        reg_alpha=(
            validation_config.xgb_reg_alpha
        ),

        reg_lambda=(
            validation_config.xgb_reg_lambda
        ),

        random_state=(
            validation_config.random_seed
        ),

        n_jobs=-1,

        tree_method="hist"
    )

In [806]:
# ============================================================
# Walk-forward XGBoost evaluation
# ============================================================

def evaluate_xgboost_walk_forward(
    data: pd.DataFrame,
    feature_columns: List[str],
    config: ForecastConfig,
    validation_config: ValidationConfig
) -> pd.DataFrame:

    windows = create_validation_windows(
        data[config.date_col],
        validation_config
    )

    results = []

    for window_number, (
        train_end,
        validation_end
    ) in enumerate(windows, start=1):

        logger.info(
            "Validation window %s | "
            "train_end=%s | validation_end=%s",
            window_number,
            train_end.strftime("%Y-%m"),
            validation_end.strftime("%Y-%m")
        )

        train_df, validation_df = temporal_split(
            data,
            train_end,
            validation_end,
            config
        )

        # Remove rows where target is unavailable
        train_df = train_df[
            train_df[config.target_col].notna()
        ]

        validation_df = validation_df[
            validation_df[config.target_col].notna()
        ]

        if train_df.empty or validation_df.empty:
            continue

        # ----------------------------------------------
        # Training data
        # ----------------------------------------------

        X_train, y_train = build_xy(
            train_df,
            feature_columns,
            config
        )

        # ----------------------------------------------
        # Validation data
        # ----------------------------------------------

        X_valid, y_valid = build_xy(
            validation_df,
            feature_columns,
            config
        )

        # ----------------------------------------------
        # Model
        # ----------------------------------------------

        model = create_baseline_xgb(
            validation_config
        )

        start_time = time.time()

        model.fit(
            X_train,
            y_train,
            verbose=False
        )

        training_seconds = (
            time.time() - start_time
        )

        predictions = model.predict(
            X_valid
        )

        if config.enforce_non_negative_forecast:

            predictions = np.maximum(
                predictions,
                0
            )

        metrics = calculate_metrics(
            y_valid.to_numpy(),
            predictions
        )

        results.append({

            "WINDOW": window_number,

            "TRAIN_END": train_end,

            "VALIDATION_END": validation_end,

            "TRAIN_ROWS": len(train_df),

            "VALIDATION_ROWS": len(validation_df),

            "TRAINING_SECONDS": training_seconds,

            "MODEL": "XGBoost",

            **metrics
        })

    return pd.DataFrame(results)

In [807]:
# ============================================================
# Run XGBoost walk-forward validation
# ============================================================

xgb_validation_results = (
    evaluate_xgboost_walk_forward(
        model_df,
        FEATURE_COLUMNS,
        CONFIG,
        VALIDATION_CONFIG
    )
)

display(
    xgb_validation_results
)

2026-08-30 21:37:39,383 | INFO | Validation window 1 | train_end=2026-02 | validation_end=2026-03
2026-08-30 21:37:43,252 | INFO | Validation window 2 | train_end=2026-03 | validation_end=2026-04
2026-08-30 21:37:44,282 | INFO | Validation window 3 | train_end=2026-04 | validation_end=2026-05
2026-08-30 21:37:45,254 | INFO | Validation window 4 | train_end=2026-05 | validation_end=2026-06
2026-08-30 21:37:46,237 | INFO | Validation window 5 | train_end=2026-06 | validation_end=2026-07


,WINDOW,TRAIN_END,VALIDATION_END,TRAIN_ROWS,VALIDATION_ROWS,TRAINING_SECONDS,MODEL,MAE,RMSE,WAPE,sMAPE,MAPE,R2
0,1,2026-02-01,2026-03-01,7,1,2.246551,XGBoost,19722.429688,19722.429688,0.286232,0.334038,0.286232,0.0
1,2,2026-03-01,2026-04-01,8,1,0.983970,XGBoost,26799.386719,26799.386719,0.325552,0.388847,0.325552,0.0
2,3,2026-04-01,2026-05-01,9,1,0.952184,XGBoost,32641.250000,32641.250000,0.340952,0.411021,0.340952,0.0
3,4,2026-05-01,2026-06-01,10,1,0.960800,XGBoost,21068.968750,21068.968750,0.193025,0.213644,0.193025,0.0
4,5,2026-06-01,2026-07-01,11,1,1.033592,XGBoost,44815.523438,44815.523438,0.365639,0.447439,0.365639,0.0


In [808]:
# ============================================================
# Naive walk-forward evaluation
# ============================================================

def evaluate_naive_walk_forward(
    data: pd.DataFrame,
    config: ForecastConfig,
    validation_config: ValidationConfig
) -> pd.DataFrame:

    windows = create_validation_windows(
        data[config.date_col],
        validation_config
    )

    results = []

    group_cols = [
        config.sponsor_col,
        config.subscriber_col
    ]

    for window_number, (
        train_end,
        validation_end
    ) in enumerate(windows, start=1):

        train_df, validation_df = temporal_split(
            data,
            train_end,
            validation_end,
            config
        )

        predictions = []
        actuals = []

        for keys, valid_group in validation_df.groupby(
            group_cols
        ):

            if not isinstance(keys, tuple):
                keys = (keys,)

            sponsor = keys[0]
            subscriber = keys[1]

            history = train_df[
                (train_df[config.sponsor_col] == sponsor) &
                (train_df[config.subscriber_col] == subscriber)
            ][config.target_col]

            prediction = naive_forecast(
                history
            )

            for actual in valid_group[
                config.target_col
            ]:

                if not np.isnan(prediction):

                    predictions.append(
                        prediction
                    )

                    actuals.append(
                        actual
                    )

        if not actuals:
            continue

        actual_array = np.asarray(
            actuals
        )

        prediction_array = np.asarray(
            predictions
        )

        metrics = calculate_metrics(
            actual_array,
            prediction_array
        )

        results.append({

            "WINDOW": window_number,

            "TRAIN_END": train_end,

            "VALIDATION_END": validation_end,

            "MODEL": "Naive",

            "TRAIN_ROWS": len(train_df),

            "VALIDATION_ROWS": len(actual_array),

            **metrics
        })

    return pd.DataFrame(results)

In [809]:
naive_results = evaluate_naive_walk_forward(
    model_df,
    CONFIG,
    VALIDATION_CONFIG
)

display(
    naive_results
)

,WINDOW,TRAIN_END,VALIDATION_END,MODEL,TRAIN_ROWS,VALIDATION_ROWS,MAE,RMSE,WAPE,sMAPE,MAPE,R2
0,1,2026-02-01,2026-03-01,Naive,7,1,13316.5,13316.5,0.193262,0.213935,0.193262,0.0
1,2,2026-03-01,2026-04-01,Naive,8,1,13416.0,13416.0,0.162974,0.177433,0.162974,0.0
2,3,2026-04-01,2026-05-01,Naive,9,1,13416.0,13416.0,0.140136,0.150695,0.140136,0.0
3,4,2026-05-01,2026-06-01,Naive,10,1,13416.0,13416.0,0.122911,0.130960,0.122911,0.0
4,5,2026-06-01,2026-07-01,Naive,11,1,13416.0,13416.0,0.109458,0.115795,0.109458,0.0


In [810]:
# ============================================================
# Moving-average walk-forward evaluation
# ============================================================

def evaluate_moving_average_walk_forward(
    data: pd.DataFrame,
    config: ForecastConfig,
    validation_config: ValidationConfig
) -> pd.DataFrame:

    windows = create_validation_windows(
        data[config.date_col],
        validation_config
    )

    results = []

    group_cols = [
        config.sponsor_col,
        config.subscriber_col
    ]

    for window_number, (
        train_end,
        validation_end
    ) in enumerate(windows, start=1):

        train_df, validation_df = temporal_split(
            data,
            train_end,
            validation_end,
            config
        )

        predictions = []
        actuals = []

        for keys, valid_group in validation_df.groupby(
            group_cols
        ):

            if not isinstance(keys, tuple):
                keys = (keys,)

            sponsor = keys[0]
            subscriber = keys[1]

            history = train_df[
                (train_df[config.sponsor_col] == sponsor) &
                (train_df[config.subscriber_col] == subscriber)
            ][config.target_col]

            prediction = moving_average_forecast(
                history,
                validation_config.moving_average_window
            )

            for actual in valid_group[
                config.target_col
            ]:

                if not np.isnan(prediction):

                    predictions.append(
                        prediction
                    )

                    actuals.append(
                        actual
                    )

        if not actuals:
            continue

        metrics = calculate_metrics(
            np.asarray(actuals),
            np.asarray(predictions)
        )

        results.append({

            "WINDOW": window_number,

            "TRAIN_END": train_end,

            "VALIDATION_END": validation_end,

            "MODEL": "MovingAverage",

            "TRAIN_ROWS": len(train_df),

            "VALIDATION_ROWS": len(actuals),

            **metrics
        })

    return pd.DataFrame(results)

In [811]:
moving_average_results = (
    evaluate_moving_average_walk_forward(
        model_df,
        CONFIG,
        VALIDATION_CONFIG
    )
)

display(
    moving_average_results
)

,WINDOW,TRAIN_END,VALIDATION_END,MODEL,TRAIN_ROWS,VALIDATION_ROWS,MAE,RMSE,WAPE,sMAPE,MAPE,R2
0,1,2026-02-01,2026-03-01,MovingAverage,7,1,25645.140000,25645.140000,0.372188,0.457286,0.372188,0.0
1,2,2026-03-01,2026-04-01,MovingAverage,8,1,26802.330000,26802.330000,0.325588,0.388898,0.325588,0.0
2,3,2026-04-01,2026-05-01,MovingAverage,9,1,26798.833333,26798.833333,0.279925,0.325480,0.279925,0.0
3,4,2026-05-01,2026-06-01,MovingAverage,10,1,26832.000000,26832.000000,0.245823,0.280271,0.245823,0.0
4,5,2026-06-01,2026-07-01,MovingAverage,11,1,26832.000000,26832.000000,0.218916,0.245823,0.218916,0.0


In [812]:
# ============================================================
# Combine model results
# ============================================================

all_validation_results = pd.concat(
    [
        naive_results,
        moving_average_results,
        xgb_validation_results
    ],
    ignore_index=True
)

display(
    all_validation_results
)

,WINDOW,TRAIN_END,VALIDATION_END,MODEL,TRAIN_ROWS,VALIDATION_ROWS,MAE,RMSE,WAPE,sMAPE,MAPE,R2,TRAINING_SECONDS
0,1,2026-02-01,2026-03-01,Naive,7,1,13316.500000,13316.500000,0.193262,0.213935,0.193262,0.0,NaN
1,2,2026-03-01,2026-04-01,Naive,8,1,13416.000000,13416.000000,0.162974,0.177433,0.162974,0.0,NaN
2,3,2026-04-01,2026-05-01,Naive,9,1,13416.000000,13416.000000,0.140136,0.150695,0.140136,0.0,NaN
3,4,2026-05-01,2026-06-01,Naive,10,1,13416.000000,13416.000000,0.122911,0.130960,0.122911,0.0,NaN
4,5,2026-06-01,2026-07-01,Naive,11,1,13416.000000,13416.000000,0.109458,0.115795,0.109458,0.0,NaN
5,1,2026-02-01,2026-03-01,MovingAverage,7,1,25645.140000,25645.140000,0.372188,0.457286,0.372188,0.0,NaN
6,2,2026-03-01,2026-04-01,MovingAverage,8,1,26802.330000,26802.330000,0.325588,0.388898,0.325588,0.0,NaN
7,3,2026-04-01,2026-05-01,MovingAverage,9,1,26798.833333,26798.833333,0.279925,0.325480,0.279925,0.0,NaN
8,4,2026-05-01,2026-06-01,MovingAverage,10,1,26832.000000,26832.000000,0.245823,0.280271,0.245823,0.0,NaN
9,5,2026-06-01,2026-07-01,MovingAverage,11,1,26832.000000,26832.000000,0.218916,0.245823,0.218916,0.0,NaN


In [813]:
# ============================================================
# Model comparison
# ============================================================

model_comparison = (
    all_validation_results
    .groupby("MODEL")
    .agg(
        MAE=("MAE", "mean"),
        RMSE=("RMSE", "mean"),
        WAPE=("WAPE", "mean"),
        sMAPE=("sMAPE", "mean"),
        MAPE=("MAPE", "mean"),
        R2=("R2", "mean"),
        VALIDATION_WINDOWS=("WINDOW", "nunique")
    )
    .reset_index()
)

model_comparison = model_comparison.sort_values(
    "WAPE"
)

display(
    model_comparison
)

,MODEL,MAE,RMSE,WAPE,sMAPE,MAPE,R2,VALIDATION_WINDOWS
1,Naive,13396.100000,13396.100000,0.145748,0.157763,0.145748,0.0,5
0,MovingAverage,26582.060667,26582.060667,0.288488,0.339552,0.288488,0.0,5
2,XGBoost,29009.511719,29009.511719,0.302280,0.358998,0.302280,0.0,5


In [814]:
# ============================================================
# Select best model based on WAPE
# ============================================================

best_model_row = (
    model_comparison
    .sort_values("WAPE")
    .iloc[0]
)

print(
    "Best validation model:",
    best_model_row["MODEL"]
)

print(
    "Average WAPE:",
    f"{best_model_row['WAPE']:.4f}"
)

Best validation model: Naive
Average WAPE: 0.1457


In [815]:
# ============================================================
# XGBoost improvement over Naive
# ============================================================

naive_wape = model_comparison.loc[
    model_comparison["MODEL"] == "Naive",
    "WAPE"
].iloc[0]

xgb_wape = model_comparison.loc[
    model_comparison["MODEL"] == "XGBoost",
    "WAPE"
].iloc[0]

improvement = (
    (naive_wape - xgb_wape)
    / naive_wape
)

print(
    f"XGBoost WAPE improvement over Naive: "
    f"{improvement:.2%}"
)

XGBoost WAPE improvement over Naive: -107.40%


In [816]:
# ============================================================
# Subscriber-level XGBoost validation
# ============================================================

def evaluate_subscriber_level(
    data: pd.DataFrame,
    feature_columns: List[str],
    config: ForecastConfig,
    validation_config: ValidationConfig
) -> pd.DataFrame:

    windows = create_validation_windows(
        data[config.date_col],
        validation_config
    )

    results = []

    group_cols = [
        config.sponsor_col,
        config.subscriber_col
    ]

    for window_number, (
        train_end,
        validation_end
    ) in enumerate(windows, start=1):

        train_df, validation_df = temporal_split(
            data,
            train_end,
            validation_end,
            config
        )

        train_df = train_df[
            train_df[config.target_col].notna()
        ]

        validation_df = validation_df[
            validation_df[config.target_col].notna()
        ]

        if train_df.empty or validation_df.empty:
            continue

        X_train, y_train = build_xy(
            train_df,
            feature_columns,
            config
        )

        X_valid, y_valid = build_xy(
            validation_df,
            feature_columns,
            config
        )

        model = create_baseline_xgb(
            validation_config
        )

        model.fit(
            X_train,
            y_train,
            verbose=False
        )

        validation_df = validation_df.copy()

        validation_df["PREDICTION"] = (
            model.predict(X_valid)
        )

        if config.enforce_non_negative_forecast:

            validation_df["PREDICTION"] = (
                validation_df["PREDICTION"]
                .clip(lower=0)
            )

        for keys, group in validation_df.groupby(
            group_cols
        ):

            metrics = calculate_metrics(
                group[
                    config.target_col
                ].to_numpy(),

                group[
                    "PREDICTION"
                ].to_numpy()
            )

            results.append({

                config.sponsor_col: keys[0],

                config.subscriber_col: keys[1],

                "WINDOW": window_number,

                "TRAIN_END": train_end,

                "VALIDATION_END": validation_end,

                **metrics
            })

    return pd.DataFrame(results)

In [817]:
subscriber_validation_results = (
    evaluate_subscriber_level(
        model_df,
        FEATURE_COLUMNS,
        CONFIG,
        VALIDATION_CONFIG
    )
)

display(
    subscriber_validation_results.head(20)
)

,SPSR_ID,SBSR_ID,WINDOW,TRAIN_END,VALIDATION_END,MAE,RMSE,WAPE,sMAPE,MAPE,R2
0,AG0025,AG0025S100000097,1,2026-02-01,2026-03-01,19722.429688,19722.429688,0.286232,0.334038,0.286232,0.0
1,AG0025,AG0025S100000097,2,2026-03-01,2026-04-01,26799.386719,26799.386719,0.325552,0.388847,0.325552,0.0
2,AG0025,AG0025S100000097,3,2026-04-01,2026-05-01,32641.250000,32641.250000,0.340952,0.411021,0.340952,0.0
3,AG0025,AG0025S100000097,4,2026-05-01,2026-06-01,21068.968750,21068.968750,0.193025,0.213644,0.193025,0.0
4,AG0025,AG0025S100000097,5,2026-06-01,2026-07-01,44815.523438,44815.523438,0.365639,0.447439,0.365639,0.0


In [818]:
# ============================================================
# Subscriber-level performance summary
# ============================================================

subscriber_summary = (
    subscriber_validation_results
    .groupby(
        [
            CONFIG.sponsor_col,
            CONFIG.subscriber_col
        ]
    )
    .agg(
        AVG_MAE=("MAE", "mean"),
        AVG_WAPE=("WAPE", "mean"),
        AVG_RMSE=("RMSE", "mean"),
        VALIDATION_WINDOWS=("WINDOW", "nunique")
    )
    .reset_index()
)

display(
    subscriber_summary.sort_values(
        "AVG_WAPE",
        ascending=False
    ).head(20)
)

,SPSR_ID,SBSR_ID,AVG_MAE,AVG_WAPE,AVG_RMSE,VALIDATION_WINDOWS
0,AG0025,AG0025S100000097,29009.511719,0.30228,29009.511719,5


In [819]:
# ============================================================
# Train diagnostic XGBoost model
# ============================================================

X_all, y_all = build_xy(
    model_df,
    FEATURE_COLUMNS,
    CONFIG
)

diagnostic_model = create_baseline_xgb(
    VALIDATION_CONFIG
)

diagnostic_model.fit(
    X_all,
    y_all,
    verbose=False
)

feature_importance = pd.DataFrame({
    "FEATURE": FEATURE_COLUMNS,
    "IMPORTANCE": diagnostic_model.feature_importances_
})

feature_importance = (
    feature_importance
    .sort_values(
        "IMPORTANCE",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    feature_importance.head(30)
)

,FEATURE,IMPORTANCE
0,QUARTER_COS,0.637688
1,YEAR,0.107649
2,LAG_1,0.085360
3,MONTH,0.025206
4,LAG_6,0.022772
5,MONTH_SIN,0.021529
6,LAG_2,0.018743
7,MONTH_COS,0.018188
8,LAG_3,0.015960
9,VOLATILITY_6M,0.012004


In [820]:
# ============================================================
# XGBoost with log1p target
# ============================================================

def evaluate_xgb_log_target(
    data: pd.DataFrame,
    feature_columns: List[str],
    config: ForecastConfig,
    validation_config: ValidationConfig
) -> pd.DataFrame:

    windows = create_validation_windows(
        data[config.date_col],
        validation_config
    )

    results = []

    for window_number, (
        train_end,
        validation_end
    ) in enumerate(windows, start=1):

        train_df, validation_df = temporal_split(
            data,
            train_end,
            validation_end,
            config
        )

        train_df = train_df[
            train_df[config.target_col].notna()
        ]

        validation_df = validation_df[
            validation_df[config.target_col].notna()
        ]

        if train_df.empty or validation_df.empty:
            continue

        X_train, y_train = build_xy(
            train_df,
            feature_columns,
            config
        )

        X_valid, y_valid = build_xy(
            validation_df,
            feature_columns,
            config
        )

        # MONEY_OUT should normally be non-negative.
        if (y_train < 0).any():

            logger.warning(
                "Negative MONEY_OUT found. "
                "Skipping log-target window."
            )

            continue

        y_train_log = np.log1p(
            y_train
        )

        model = create_baseline_xgb(
            validation_config
        )

        model.fit(
            X_train,
            y_train_log,
            verbose=False
        )

        prediction_log = model.predict(
            X_valid
        )

        predictions = np.expm1(
            prediction_log
        )

        predictions = np.maximum(
            predictions,
            0
        )

        metrics = calculate_metrics(
            y_valid.to_numpy(),
            predictions
        )

        results.append({

            "WINDOW": window_number,

            "TRAIN_END": train_end,

            "VALIDATION_END": validation_end,

            "MODEL": "XGBoost_log1p",

            **metrics
        })

    return pd.DataFrame(results)

In [821]:
xgb_log_results = (
    evaluate_xgb_log_target(
        model_df,
        FEATURE_COLUMNS,
        CONFIG,
        VALIDATION_CONFIG
    )
)

display(
    xgb_log_results
)

,WINDOW,TRAIN_END,VALIDATION_END,MODEL,MAE,RMSE,WAPE,sMAPE,MAPE,R2
0,1,2026-02-01,2026-03-01,XGBoost_log1p,40786.503906,40786.503906,0.591934,0.840777,0.591934,0.0
1,2,2026-03-01,2026-04-01,XGBoost_log1p,61624.833984,61624.833984,0.748603,1.196428,0.748603,0.0
2,3,2026-04-01,2026-05-01,XGBoost_log1p,62975.169922,62975.169922,0.657802,0.980186,0.657802,0.0
3,4,2026-05-01,2026-06-01,XGBoost_log1p,4188.046875,4188.046875,0.038369,0.039120,0.038369,0.0
4,5,2026-06-01,2026-07-01,XGBoost_log1p,94006.261719,94006.261719,0.766974,1.244051,0.766974,0.0


In [822]:
# ============================================================
# Raw vs log target
# ============================================================

target_comparison = pd.concat(
    [
        xgb_validation_results,
        xgb_log_results
    ],
    ignore_index=True
)

target_summary = (
    target_comparison
    .groupby("MODEL")
    .agg(
        MAE=("MAE", "mean"),
        RMSE=("RMSE", "mean"),
        WAPE=("WAPE", "mean"),
        sMAPE=("sMAPE", "mean"),
        MAPE=("MAPE", "mean"),
        R2=("R2", "mean")
    )
    .reset_index()
    .sort_values("WAPE")
)

display(
    target_summary
)

,MODEL,MAE,RMSE,WAPE,sMAPE,MAPE,R2
0,XGBoost,29009.511719,29009.511719,0.302280,0.358998,0.302280,0.0
1,XGBoost_log1p,52716.163281,52716.163281,0.560737,0.860112,0.560737,0.0


In [823]:
%pip install optuna

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\udayn\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [824]:
import optuna

from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

from xgboost import XGBRegressor

logger.info("Optuna and XGBoost initialized.")

2026-08-30 21:38:02,864 | INFO | Optuna and XGBoost initialized.


In [825]:
from dataclasses import dataclass


@dataclass
class OptimizationConfig:

    # Number of Optuna trials
    n_trials: int = 50

    # Optimization metric
    primary_metric: str = "WAPE"

    # Random seed
    random_seed: int = 42

    # XGBoost objective
    objective: str = "reg:squarederror"

    # Number of CPU threads
    n_jobs: int = -1

    # Early stopping
    early_stopping_rounds: int = 30

    # XGBoost parameter ranges
    min_estimators: int = 100
    max_estimators: int = 1000


OPTIMIZATION_CONFIG = OptimizationConfig()

In [826]:
optimization_windows = create_validation_windows(
    model_df[CONFIG.date_col],
    VALIDATION_CONFIG
)

print("Optimization windows:")

for i, (
    train_end,
    validation_end
) in enumerate(
    optimization_windows,
    start=1
):

    print(
        f"Window {i}: "
        f"{train_end.strftime('%Y-%m')} "
        f"-> "
        f"{validation_end.strftime('%Y-%m')}"
    )

Optimization windows:
Window 1: 2026-02 -> 2026-03
Window 2: 2026-03 -> 2026-04
Window 3: 2026-04 -> 2026-05
Window 4: 2026-05 -> 2026-06
Window 5: 2026-06 -> 2026-07


In [827]:
def get_xgb_params(
    trial: optuna.Trial,
    optimization_config: OptimizationConfig
) -> dict:

    params = {

        "objective":
            optimization_config.objective,

        "n_estimators":
            trial.suggest_int(
                "n_estimators",
                optimization_config.min_estimators,
                optimization_config.max_estimators
            ),

        "learning_rate":
            trial.suggest_float(
                "learning_rate",
                0.01,
                0.20,
                log=True
            ),

        "max_depth":
            trial.suggest_int(
                "max_depth",
                2,
                8
            ),

        "min_child_weight":
            trial.suggest_int(
                "min_child_weight",
                1,
                20
            ),

        "subsample":
            trial.suggest_float(
                "subsample",
                0.60,
                1.00
            ),

        "colsample_bytree":
            trial.suggest_float(
                "colsample_bytree",
                0.60,
                1.00
            ),

        "gamma":
            trial.suggest_float(
                "gamma",
                0.0,
                5.0
            ),

        "reg_alpha":
            trial.suggest_float(
                "reg_alpha",
                1e-8,
                10.0,
                log=True
            ),

        "reg_lambda":
            trial.suggest_float(
                "reg_lambda",
                0.01,
                20.0,
                log=True
            ),

        "random_state":
            optimization_config.random_seed,

        "n_jobs":
            optimization_config.n_jobs,

        "tree_method":
            "hist"
    }

    return params

In [828]:
def xgb_objective(
    trial: optuna.Trial,
    data: pd.DataFrame,
    feature_columns: List[str],
    config: ForecastConfig,
    validation_config: ValidationConfig,
    optimization_config: OptimizationConfig,
    target_transform: str = "raw"
) -> float:

    params = get_xgb_params(
        trial,
        optimization_config
    )

    window_scores = []

    windows = create_validation_windows(
        data[config.date_col],
        validation_config
    )

    for window_number, (
        train_end,
        validation_end
    ) in enumerate(windows, start=1):

        train_df, validation_df = temporal_split(
            data,
            train_end,
            validation_end,
            config
        )

        train_df = train_df[
            train_df[config.target_col].notna()
        ]

        validation_df = validation_df[
            validation_df[config.target_col].notna()
        ]

        if train_df.empty or validation_df.empty:
            continue

        X_train, y_train = build_xy(
            train_df,
            feature_columns,
            config
        )

        X_valid, y_valid = build_xy(
            validation_df,
            feature_columns,
            config
        )

        # ------------------------------------------
        # Target transformation
        # ------------------------------------------

        if target_transform == "log1p":

            if (y_train < 0).any():

                return float("inf")

            y_train_model = np.log1p(
                y_train
            )

        else:

            y_train_model = y_train

        # ------------------------------------------
        # Model
        # ------------------------------------------

        model = XGBRegressor(
            **params
        )

        # ------------------------------------------
        # Train
        # ------------------------------------------

        model.fit(
            X_train,
            y_train_model,
            eval_set=[
                (X_valid, (
                    np.log1p(y_valid)
                    if target_transform == "log1p"
                    else y_valid
                ))
            ],
            verbose=False
        )

        # ------------------------------------------
        # Prediction
        # ------------------------------------------

        predictions = model.predict(
            X_valid
        )

        if target_transform == "log1p":

            predictions = np.expm1(
                predictions
            )

        predictions = np.maximum(
            predictions,
            0
        )

        # ------------------------------------------
        # Metrics
        # ------------------------------------------

        metrics = calculate_metrics(
            y_valid.to_numpy(),
            predictions
        )

        score = metrics[
            optimization_config.primary_metric
        ]

        if np.isfinite(score):

            window_scores.append(
                score
            )

        # ------------------------------------------
        # Optuna pruning
        # ------------------------------------------

        if window_scores:

            intermediate_score = float(
                np.mean(window_scores)
            )

            trial.report(
                intermediate_score,
                step=window_number
            )

            if trial.should_prune():

                raise optuna.TrialPruned()

    if not window_scores:

        return float("inf")

    return float(
        np.mean(window_scores)
    )

In [829]:
def create_optuna_study(
    study_name: str,
    optimization_config: OptimizationConfig
) -> optuna.Study:

    sampler = TPESampler(
        seed=optimization_config.random_seed
    )

    pruner = MedianPruner(
        n_startup_trials=5,
        n_warmup_steps=1
    )

    study = optuna.create_study(
        study_name=study_name,
        direction="minimize",
        sampler=sampler,
        pruner=pruner
    )

    return study

In [830]:
raw_study = create_optuna_study(
    study_name="money_out_xgb_raw",
    optimization_config=OPTIMIZATION_CONFIG
)

raw_study.optimize(
    lambda trial: xgb_objective(
        trial=trial,
        data=model_df,
        feature_columns=FEATURE_COLUMNS,
        config=CONFIG,
        validation_config=VALIDATION_CONFIG,
        optimization_config=OPTIMIZATION_CONFIG,
        target_transform="raw"
    ),
    n_trials=OPTIMIZATION_CONFIG.n_trials,
    show_progress_bar=True
)

[I 2026-08-30 21:38:03,627] A new study created in memory with name: money_out_xgb_raw
Best trial: 0. Best value: 0.620661:   2%|▏         | 1/50 [00:13<11:02, 13.51s/it]

[I 2026-08-30 21:38:18,015] Trial 0 finished with value: 0.620661208595797 and parameters: {'n_estimators': 437, 'learning_rate': 0.17254716573280354, 'max_depth': 7, 'min_child_weight': 12, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.662397808134481, 'gamma': 0.2904180608409973, 'reg_alpha': 0.6245760287469893, 'reg_lambda': 0.9644921218755025}. Best is trial 0 with value: 0.620661208595797.


Best trial: 0. Best value: 0.620661:   4%|▍         | 2/50 [00:32<13:23, 16.74s/it]

[I 2026-08-30 21:38:37,134] Trial 1 finished with value: 0.6232884350816016 and parameters: {'n_estimators': 737, 'learning_rate': 0.010636066512540286, 'max_depth': 8, 'min_child_weight': 17, 'subsample': 0.6849356442713105, 'colsample_bytree': 0.6727299868828402, 'gamma': 0.9170225492671691, 'reg_alpha': 5.472429642032198e-06, 'reg_lambda': 0.5398047741548972}. Best is trial 0 with value: 0.620661208595797.


Best trial: 2. Best value: 0.297661:   6%|▌         | 3/50 [00:45<11:48, 15.08s/it]

...

[I 2026-08-30 21:42:10,509] Trial 47 pruned. 


Best trial: 14. Best value: 0.208244:  98%|█████████▊| 49/50 [04:07<00:01,  1.78s/it]

[I 2026-08-30 21:42:12,378] Trial 48 pruned. 


Best trial: 14. Best value: 0.208244: 100%|██████████| 50/50 [04:08<00:00,  4.98s/it]

[I 2026-08-30 21:42:13,381] Trial 49 pruned. 


In [831]:
print(
    "Best raw MONEY_OUT WAPE:",
    raw_study.best_value
)

print("\nBest parameters:")

for key, value in raw_study.best_params.items():

    print(
        f"{key}: {value}"
    )

Best raw MONEY_OUT WAPE: 0.20824391376139229

Best parameters:
n_estimators: 111
learning_rate: 0.05160037849149582
max_depth: 3
min_child_weight: 1
subsample: 0.9997831683127447
colsample_bytree: 0.8183562936796783
gamma: 3.759064545366492
reg_alpha: 0.00015859927884178952
reg_lambda: 0.012123187152476223


In [832]:
log_study = create_optuna_study(
    study_name="money_out_xgb_log1p",
    optimization_config=OPTIMIZATION_CONFIG
)

log_study.optimize(
    lambda trial: xgb_objective(
        trial=trial,
        data=model_df,
        feature_columns=FEATURE_COLUMNS,
        config=CONFIG,
        validation_config=VALIDATION_CONFIG,
        optimization_config=OPTIMIZATION_CONFIG,
        target_transform="log1p"
    ),
    n_trials=OPTIMIZATION_CONFIG.n_trials,
    show_progress_bar=True
)

[I 2026-08-30 21:42:13,655] A new study created in memory with name: money_out_xgb_log1p
Best trial: 0. Best value: 0.803735:   2%|▏         | 1/50 [00:11<09:38, 11.81s/it]

[I 2026-08-30 21:42:25,466] Trial 0 finished with value: 0.8037352975049554 and parameters: {'n_estimators': 437, 'learning_rate': 0.17254716573280354, 'max_depth': 7, 'min_child_weight': 12, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.662397808134481, 'gamma': 0.2904180608409973, 'reg_alpha': 0.6245760287469893, 'reg_lambda': 0.9644921218755025}. Best is trial 0 with value: 0.8037352975049554.


Best trial: 1. Best value: 0.799014:   4%|▍         | 2/50 [00:36<15:24, 19.26s/it]

[I 2026-08-30 21:42:49,949] Trial 1 finished with value: 0.7990139262218079 and parameters: {'n_estimators': 737, 'learning_rate': 0.010636066512540286, 'max_depth': 8, 'min_child_weight': 17, 'subsample': 0.6849356442713105, 'colsample_bytree': 0.6727299868828402, 'gamma': 0.9170225492671691, 'reg_alpha': 5.472429642032198e-06, 'reg_lambda': 0.5398047741548972}. Best is trial 1 with value: 0.7990139262218079.


Best trial: 2. Best value: 0.652119:   6%|▌         | 3/50 [00:56<15:17, 19.53s/it]

...

[I 2026-08-30 21:46:45,300] Trial 47 pruned. 


Best trial: 42. Best value: 0.615787:  98%|█████████▊| 49/50 [04:32<00:02,  2.04s/it]

[I 2026-08-30 21:46:45,986] Trial 48 pruned. 


Best trial: 42. Best value: 0.615787: 100%|██████████| 50/50 [04:34<00:00,  5.48s/it]

[I 2026-08-30 21:46:47,695] Trial 49 pruned. 


In [833]:
raw_wape = raw_study.best_value
log_wape = log_study.best_value

print(
    f"Raw XGBoost WAPE:    {raw_wape:.4%}"
)

print(
    f"log1p XGBoost WAPE:  {log_wape:.4%}"
)

if log_wape < raw_wape:

    selected_target_transform = "log1p"

    print(
        "\nSelected target transformation: log1p"
    )

else:

    selected_target_transform = "raw"

    print(
        "\nSelected target transformation: raw"
    )

Raw XGBoost WAPE:    20.8244%
log1p XGBoost WAPE:  61.5787%

Selected target transformation: raw


In [834]:
if selected_target_transform == "log1p":

    best_params = log_study.best_params

else:

    best_params = raw_study.best_params


print(
    "Selected target transformation:",
    selected_target_transform
)

print("\nSelected parameters:")

for key, value in best_params.items():

    print(
        f"{key}: {value}"
    )

Selected target transformation: raw

Selected parameters:
n_estimators: 111
learning_rate: 0.05160037849149582
max_depth: 3
min_child_weight: 1
subsample: 0.9997831683127447
colsample_bytree: 0.8183562936796783
gamma: 3.759064545366492
reg_alpha: 0.00015859927884178952
reg_lambda: 0.012123187152476223


In [835]:
def train_final_xgb(
    data: pd.DataFrame,
    feature_columns: List[str],
    config: ForecastConfig,
    best_params: Dict,
    target_transform: str
) -> XGBRegressor:

    data = data.copy()

    data = data[
        data[config.target_col].notna()
    ]

    X, y = build_xy(
        data,
        feature_columns,
        config
    )

    if target_transform == "log1p":

        if (y < 0).any():

            raise ValueError(
                "Negative MONEY_OUT values "
                "cannot be used with log1p."
            )

        y_model = np.log1p(y)

    else:

        y_model = y

    model = XGBRegressor(
        objective="reg:squarederror",

        **best_params,

        random_state=OPTIMIZATION_CONFIG.random_seed,

        n_jobs=OPTIMIZATION_CONFIG.n_jobs,

        tree_method="hist"
    )

    logger.info(
        "Training final optimized XGBoost model."
    )

    model.fit(
        X,
        y_model,
        verbose=False
    )

    return model

In [836]:
final_xgb_model = train_final_xgb(
    data=model_df,
    feature_columns=FEATURE_COLUMNS,
    config=CONFIG,
    best_params=best_params,
    target_transform=selected_target_transform
)

print(
    "Final optimized XGBoost model trained."
)

2026-08-30 21:46:47,916 | INFO | Training final optimized XGBoost model.


Final optimized XGBoost model trained.


In [837]:
optimized_feature_importance = pd.DataFrame({

    "FEATURE":
        FEATURE_COLUMNS,

    "IMPORTANCE":
        final_xgb_model.feature_importances_

})

optimized_feature_importance = (
    optimized_feature_importance
    .sort_values(
        "IMPORTANCE",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    optimized_feature_importance.head(30)
)

,FEATURE,IMPORTANCE
0,LAG_2,0.238055
1,LAG_4,0.233011
2,LAG_1,0.208299
3,LAG_3,0.203253
4,ROLLING_STD_2,0.023756
5,LAG_6,0.016975
6,ROLLING_STD_3,0.016788
7,MONTH_NUMBER,0.011397
8,YEAR,0.011130
9,MONTH_COS,0.011040


In [838]:
model_metadata = {

    "MODEL_NAME":
        "MONEY_OUT_XGBOOST",

    "MODEL_TYPE":
        "XGBRegressor",

    "TARGET":
        CONFIG.target_col,

    "TARGET_TRANSFORM":
        selected_target_transform,

    "PRIMARY_METRIC":
        OPTIMIZATION_CONFIG.primary_metric,

    "BEST_VALIDATION_WAPE":
        (
            log_study.best_value
            if selected_target_transform == "log1p"
            else raw_study.best_value
        ),

    "N_FEATURES":
        len(FEATURE_COLUMNS),

    "N_TRAINING_ROWS":
        len(model_df),

    "TRAIN_START":
        model_df[
            CONFIG.date_col
        ].min(),

    "TRAIN_END":
        model_df[
            CONFIG.date_col
        ].max(),

    "RANDOM_SEED":
        OPTIMIZATION_CONFIG.random_seed,

    "OPTUNA_TRIALS":
        OPTIMIZATION_CONFIG.n_trials,

    "FEATURES":
        FEATURE_COLUMNS,

    "BEST_PARAMETERS":
        best_params
}

In [839]:
print(
    "=============================================="
)

print(
    "FINAL MODEL"
)

print(
    "=============================================="
)

print(
    "Model:",
    model_metadata["MODEL_NAME"]
)

print(
    "Target:",
    model_metadata["TARGET"]
)

print(
    "Transformation:",
    model_metadata["TARGET_TRANSFORM"]
)

print(
    "Validation WAPE:",
    f"{model_metadata['BEST_VALIDATION_WAPE']:.4%}"
)

print(
    "Training rows:",
    model_metadata["N_TRAINING_ROWS"]
)

print(
    "Number of features:",
    model_metadata["N_FEATURES"]
)

print(
    "Training period:",
    model_metadata["TRAIN_START"],
    "to",
    model_metadata["TRAIN_END"]
)

FINAL MODEL
Model: MONEY_OUT_XGBOOST
Target: AMOUNT
Transformation: raw
Validation WAPE: 20.8244%
Training rows: 12
Number of features: 56
Training period: 2025-08-01 00:00:00 to 2026-07-01 00:00:00


In [840]:
optimized_window_results = []

for window_number, (
    train_end,
    validation_end
) in enumerate(
    optimization_windows,
    start=1
):

    train_df, validation_df = temporal_split(
        model_df,
        train_end,
        validation_end,
        CONFIG
    )

    if train_df.empty or validation_df.empty:
        continue

    X_train, y_train = build_xy(
        train_df,
        FEATURE_COLUMNS,
        CONFIG
    )

    X_valid, y_valid = build_xy(
        validation_df,
        FEATURE_COLUMNS,
        CONFIG
    )

    if selected_target_transform == "log1p":

        y_train_model = np.log1p(
            y_train
        )

    else:

        y_train_model = y_train

    model = XGBRegressor(
        objective="reg:squarederror",
        **best_params,
        random_state=OPTIMIZATION_CONFIG.random_seed,
        n_jobs=OPTIMIZATION_CONFIG.n_jobs,
        tree_method="hist"
    )

    model.fit(
        X_train,
        y_train_model,
        verbose=False
    )

    predictions = model.predict(
        X_valid
    )

    if selected_target_transform == "log1p":

        predictions = np.expm1(
            predictions
        )

    predictions = np.maximum(
        predictions,
        0
    )

    metrics = calculate_metrics(
        y_valid.to_numpy(),
        predictions
    )

    optimized_window_results.append({

        "WINDOW":
            window_number,

        "TRAIN_END":
            train_end,

        "VALIDATION_END":
            validation_end,

        "MAE":
            metrics["MAE"],

        "RMSE":
            metrics["RMSE"],

        "WAPE":
            metrics["WAPE"],

        "sMAPE":
            metrics["sMAPE"],

        "MAPE":
            metrics["MAPE"],

        "R2":
            metrics["R2"]
    })


optimized_window_results_df = pd.DataFrame(
    optimized_window_results
)

display(
    optimized_window_results_df
)

,WINDOW,TRAIN_END,VALIDATION_END,MAE,RMSE,WAPE,sMAPE,MAPE,R2
0,1,2026-02-01,2026-03-01,13399.210938,13399.210938,0.194463,0.215407,0.194463,0.0
1,2,2026-03-01,2026-04-01,13653.218750,13653.218750,0.165856,0.180854,0.165856,0.0
2,3,2026-04-01,2026-05-01,42546.671875,42546.671875,0.444418,0.571385,0.444418,0.0
3,4,2026-05-01,2026-06-01,13570.359375,13570.359375,0.124326,0.132566,0.124326,0.0
4,5,2026-06-01,2026-07-01,13746.890625,13746.890625,0.112157,0.118821,0.112157,0.0


In [841]:
optimized_average_metrics = (
    optimized_window_results_df[
        [
            "MAE",
            "RMSE",
            "WAPE",
            "sMAPE",
            "MAPE",
            "R2"
        ]
    ]
    .mean()
)

optimized_summary = pd.DataFrame({

    "MODEL": [
        "Naive",
        "MovingAverage",
        "Baseline_XGBoost",
        "Optimized_XGBoost"
    ],

    "WAPE": [
        naive_results["WAPE"].mean(),

        moving_average_results[
            "WAPE"
        ].mean(),

        xgb_validation_results[
            "WAPE"
        ].mean(),

        optimized_average_metrics[
            "WAPE"
        ]
    ]
})

display(
    optimized_summary.sort_values(
        "WAPE"
    )
)

,MODEL,WAPE
0,Naive,0.145748
3,Optimized_XGBoost,0.208244
1,MovingAverage,0.288488
2,Baseline_XGBoost,0.302280


In [842]:
baseline_wape = (
    xgb_validation_results["WAPE"]
    .mean()
)

optimized_wape = (
    optimized_average_metrics["WAPE"]
)

optimization_improvement = (
    (baseline_wape - optimized_wape)
    / baseline_wape
)

print(
    "Improvement from XGBoost tuning:",
    f"{optimization_improvement:.2%}"
)

Improvement from XGBoost tuning: 31.11%


In [843]:
from dataclasses import dataclass


@dataclass
class ForecastHorizonConfig:
    """
    Configuration for future forecasting.
    """

    horizon_months: int = 2

    minimum_history_months: int = 3

    forecast_column: str = "FORECAST_MONEY_OUT"

    forecast_month_column: str = "FORECAST_MONTH"

    model_name: str = "MONEY_OUT_XGBOOST"

    model_version: str = "1.0.0"


FORECAST_CONFIG = ForecastHorizonConfig()

In [844]:
def get_latest_historical_month(
    data: pd.DataFrame,
    date_column: str
) -> pd.Timestamp:
    """
    Return the latest month available in the dataset.
    """

    if data.empty:
        raise ValueError(
            "Input dataset is empty."
        )

    latest_month = (
        pd.to_datetime(
            data[date_column]
        )
        .dt.to_period("M")
        .max()
        .to_timestamp()
    )

    return latest_month

In [845]:
latest_month = get_latest_historical_month(
    model_df,
    CONFIG.date_col
)

print(
    "Latest historical month:",
    latest_month.strftime("%Y-%m")
)

Latest historical month: 2026-07


In [846]:
def get_forecast_months(
    latest_month: pd.Timestamp,
    horizon: int
) -> List[pd.Timestamp]:
    """
    Generate future monthly forecast dates.
    """

    return [
        (
            latest_month
            + pd.DateOffset(months=i)
        ).normalize()
        for i in range(1, horizon + 1)
    ]

In [847]:
forecast_months = get_forecast_months(
    latest_month,
    FORECAST_CONFIG.horizon_months
)

print("Forecast months:")

for month in forecast_months:
    print(
        month.strftime("%Y-%m")
    )

Forecast months:
2026-08
2026-09


In [848]:
FORECAST_KEYS = [
    "SPSR_ID",
    "SBSR_ID",
    "SUMMARY_TYPE"
    #"AMOUNT",
    #"YEAR_MONTH",
    #"TOTAL"
]



In [849]:
total_df = model_df[
    model_df["AMOUNT"]
    .astype(str)
    .str.upper()
    .eq("TOTAL")
].copy()

In [850]:
def find_duplicate_time_series_records(
    data: pd.DataFrame,
    keys: List[str],
    date_column: str
) -> pd.DataFrame:
    """
    Identify duplicate records at the forecasting grain.
    """

    duplicate_mask = (
        data
        .duplicated(
            subset=keys + [date_column],
            keep=False
        )
    )

    duplicates = (
        data.loc[duplicate_mask]
        .sort_values(
            keys + [date_column]
        )
    )

    return duplicates

In [851]:
display(FORECAST_KEYS)

['SPSR_ID', 'SBSR_ID', 'SUMMARY_TYPE']

In [852]:
duplicates = find_duplicate_time_series_records(
    model_df,
    FORECAST_KEYS,
    CONFIG.date_col
)

print(
    "Duplicate rows:",
    len(duplicates)
)

Duplicate rows: 0


In [853]:
def create_forecast_state(
    data: pd.DataFrame,
    forecast_keys: List[str],
    date_column: str,
    target_column: str
) -> Dict[tuple, pd.DataFrame]:
    """
    Create a dictionary containing the historical
    time series for every forecasting entity.
    """

    state = {}

    for entity_key, group in data.groupby(
        forecast_keys,
        dropna=False
    ):

        group = (
            group
            .sort_values(date_column)
            .copy()
        )

        state[entity_key] = group[
            forecast_keys
            + [
                date_column,
                target_column
            ]
        ].copy()

    return state

In [854]:
def create_forecast_state(
    data: pd.DataFrame,
    forecast_keys: List[str],
    date_column: str,
    target_column: str
) -> Dict[tuple, pd.DataFrame]:
    """
    Create a dictionary containing the historical
    time series for every forecasting entity.
    """

    state = {}

    for entity_key, group in data.groupby(
        forecast_keys,
        dropna=False
    ):

        group = (
            group
            .sort_values(date_column)
            .copy()
        )

        state[entity_key] = group[
            forecast_keys
            + [
                date_column,
                target_column
            ]
        ].copy()

    return state

In [855]:
forecast_state = create_forecast_state(
    model_df,
    FORECAST_KEYS,
    CONFIG.date_col,
    CONFIG.target_col
)

print(
    "Number of forecasting series:",
    len(forecast_state)
)

Number of forecasting series: 1


In [856]:
display(forecast_state)

{('AG0025',
  'AG0025S100000097',
  'MoneyOut'):    SPSR_ID           SBSR_ID SUMMARY_TYPE YEAR_MONTH     AMOUNT
 0   AG0025  AG0025S100000097     MoneyOut 2025-08-01     513.00
 1   AG0025  AG0025S100000097     MoneyOut 2025-09-01   19268.75
 2   AG0025  AG0025S100000097     MoneyOut 2025-10-01   28054.06
 3   AG0025  AG0025S100000097     MoneyOut 2025-11-01   18634.24
 4   AG0025  AG0025S100000097     MoneyOut 2025-12-01   32127.32
 5   AG0025  AG0025S100000097     MoneyOut 2026-01-01   42061.26
 6   AG0025  AG0025S100000097     MoneyOut 2026-02-01   55587.25
 7   AG0025  AG0025S100000097     MoneyOut 2026-03-01   68903.75
 8   AG0025  AG0025S100000097     MoneyOut 2026-04-01   82319.75
 9   AG0025  AG0025S100000097     MoneyOut 2026-05-01   95735.75
 10  AG0025  AG0025S100000097     MoneyOut 2026-06-01  109151.75
 11  AG0025  AG0025S100000097     MoneyOut 2026-07-01  122567.75}

In [857]:
def create_future_features(
    history: pd.DataFrame,
    forecast_month: pd.Timestamp,
    config: ForecastConfig
) -> pd.DataFrame:
    """
    Create leakage-safe features for one future month.
    """

    history = (
        history
        .sort_values(config.date_col)
        .copy()
    )

    row = {}

    # -----------------------------------------
    # Calendar features
    # -----------------------------------------

    row["YEAR"] = forecast_month.year

    row["MONTH"] = forecast_month.month

    row["MONTH_NUMBER"] = forecast_month.month

    row["QUARTER"] = (
        forecast_month.quarter
    )

    row["QUARTER_NUMBER"] = (
        forecast_month.quarter
    )

    row["IS_YEAR_START"] = int(
        forecast_month.month == 1
    )

    row["IS_YEAR_END"] = int(
        forecast_month.month == 12
    )

    row["IS_QUARTER_END"] = int(
        forecast_month.month
        in [3, 6, 9, 12]
    )

    # -----------------------------------------
    # Cyclical features
    # -----------------------------------------

    row["MONTH_SIN"] = np.sin(
        2 * np.pi * forecast_month.month / 12
    )

    row["MONTH_COS"] = np.cos(
        2 * np.pi * forecast_month.month / 12
    )

    row["QUARTER_SIN"] = np.sin(
        2 * np.pi * forecast_month.quarter / 4
    )

    row["QUARTER_COS"] = np.cos(
        2 * np.pi * forecast_month.quarter / 4
    )

    # -----------------------------------------
    # Historical target
    # -----------------------------------------

    values = (
        history[config.target_col]
        .astype(float)
        .tolist()
    )

    # -----------------------------------------
    # Lag features
    # -----------------------------------------

    lag_mapping = {
        "LAG_1": 1,
        "LAG_2": 2,
        "LAG_3": 3,
        "LAG_4": 4,
        "LAG_6": 6,
        "LAG_12": 12
    }

    for feature_name, lag in lag_mapping.items():

        if len(values) >= lag:

            row[feature_name] = (
                values[-lag]
            )

        else:

            row[feature_name] = np.nan

    # -----------------------------------------
    # Rolling features
    # -----------------------------------------

    rolling_windows = [3, 6, 12]

    for window in rolling_windows:

        recent = values[-window:]

        if len(recent) > 0:

            row[
                f"ROLLING_MEAN_{window}"
            ] = np.mean(recent)

            row[
                f"ROLLING_STD_{window}"
            ] = (
                np.std(recent)
                if len(recent) > 1
                else 0.0
            )

            row[
                f"ROLLING_MIN_{window}"
            ] = np.min(recent)

            row[
                f"ROLLING_MAX_{window}"
            ] = np.max(recent)

            row[
                f"ROLLING_SUM_{window}"
            ] = np.sum(recent)

            row[
                f"ROLLING_MEDIAN_{window}"
            ] = np.median(recent)

        else:

            row[
                f"ROLLING_MEAN_{window}"
            ] = np.nan

            row[
                f"ROLLING_STD_{window}"
            ] = np.nan

            row[
                f"ROLLING_MIN_{window}"
            ] = np.nan

            row[
                f"ROLLING_MAX_{window}"
            ] = np.nan

            row[
                f"ROLLING_SUM_{window}"
            ] = np.nan

            row[
                f"ROLLING_MEDIAN_{window}"
            ] = np.nan

    # -----------------------------------------
    # Momentum
    # -----------------------------------------

    if len(values) >= 2:

        row["MOM_CHANGE"] = (
            values[-1] - values[-2]
        )

        if values[-2] != 0:

            row["MOM_GROWTH_RATE"] = (
                (values[-1] - values[-2])
                / abs(values[-2])
            )

        else:

            row["MOM_GROWTH_RATE"] = 0.0

    else:

        row["MOM_CHANGE"] = np.nan
        row["MOM_GROWTH_RATE"] = np.nan

    # -----------------------------------------
    # Trend
    # -----------------------------------------

    if len(values) >= 3:

        row["TREND_3M"] = (
            (values[-1] - values[-3])
            / 2
        )

    else:

        row["TREND_3M"] = np.nan

    if len(values) >= 6:

        row["TREND_6M"] = (
            (values[-1] - values[-6])
            / 5
        )

    else:

        row["TREND_6M"] = np.nan

    # -----------------------------------------
    # Volatility
    # -----------------------------------------

    if len(values) >= 3:

        row["VOLATILITY_3M"] = (
            np.std(values[-3:])
        )

    else:

        row["VOLATILITY_3M"] = np.nan

    return pd.DataFrame([row])

In [858]:
def add_entity_attributes(
    feature_df: pd.DataFrame,
    history: pd.DataFrame,
    forecast_keys: List[str]
) -> pd.DataFrame:
    """
    Add entity/business attributes that are known
    at forecast time.
    """

    feature_df = feature_df.copy()

    for column in forecast_keys:

        if column in history.columns:

            feature_df[column] = (
                history[column].iloc[-1]
            )

    return feature_df

In [859]:
def predict_one_future_month(
    model: XGBRegressor,
    history: pd.DataFrame,
    forecast_month: pd.Timestamp,
    feature_columns: List[str],
    config: ForecastConfig,
    target_transform: str
) -> float:
    """
    Predict MONEY_OUT for one future month.
    """

    future_features = create_future_features(
        history=history,
        forecast_month=forecast_month,
        config=config
    )

    future_features = add_entity_attributes(
        feature_df=future_features,
        history=history,
        forecast_keys=FORECAST_KEYS
    )

    # -----------------------------------------
    # Ensure same feature order as training
    # -----------------------------------------

    X_future = future_features.reindex(
        columns=feature_columns,
        fill_value=np.nan
    )

    # -----------------------------------------
    # Predict
    # -----------------------------------------

    prediction = model.predict(
        X_future
    )[0]

    # -----------------------------------------
    # Reverse log transformation
    # -----------------------------------------

    if target_transform == "log1p":

        prediction = np.expm1(
            prediction
        )

    # MONEY_OUT cannot normally be negative
    prediction = max(
        float(prediction),
        0.0
    )

    return prediction

In [860]:
def generate_recursive_forecasts(
    model: XGBRegressor,
    data: pd.DataFrame,
    forecast_keys: List[str],
    feature_columns: List[str],
    config: ForecastConfig,
    forecast_config: ForecastHorizonConfig,
    target_transform: str
) -> pd.DataFrame:
    """
    Generate recursive forecasts for all entities.
    """

    latest_month = get_latest_historical_month(
        data,
        config.date_col
    )

    forecast_months = get_forecast_months(
        latest_month,
        forecast_config.horizon_months
    )

    state = create_forecast_state(
        data,
        forecast_keys,
        config.date_col,
        config.target_col
    )

    forecast_results = []

    for entity_key, history in state.items():

        history = history.copy()

        # -------------------------------------
        # Minimum history validation
        # -------------------------------------

        history_count_result = history[config.target_col].notna().sum()
        if isinstance(history_count_result, pd.Series):
            history_count = int(history_count_result.iloc[0]) if not history_count_result.empty else 0
        else:
            history_count = int(history_count_result)

        if history_count < forecast_config.minimum_history_months:

            logger.warning(
                "Insufficient history for entity %s: %s months",
                entity_key,
                history_count
            )

            continue

        # -------------------------------------
        # Recursive forecasting
        # -------------------------------------

        for horizon, forecast_month in enumerate(
            forecast_months,
            start=1
        ):

            prediction = predict_one_future_month(
                model=model,
                history=history,
                forecast_month=forecast_month,
                feature_columns=feature_columns,
                config=config,
                target_transform=target_transform
            )

            # ---------------------------------
            # Build output row
            # ---------------------------------

            result = {}

            for key_name, key_value in zip(
                forecast_keys,
                entity_key
            ):

                result[key_name] = key_value

            result[
                forecast_config.forecast_month_column
            ] = forecast_month

            result[
                forecast_config.forecast_column
            ] = prediction

            result[
                "MODEL_NAME"
            ] = forecast_config.model_name

            result[
                "MODEL_VERSION"
            ] = forecast_config.model_version

            result[
                "FORECAST_HORIZON"
            ] = horizon

            result[
                "DATA_HISTORY_MONTHS"
            ] = history_count

            result[
                "FORECAST_RUN_DATE"
            ] = pd.Timestamp.utcnow()

            forecast_results.append(
                result
            )

            # ---------------------------------
            # IMPORTANT:
            # Add prediction to history
            # ---------------------------------

            new_row = {
                config.date_col:
                    forecast_month,

                config.target_col:
                    prediction
            }

            for key_name, key_value in zip(
                forecast_keys,
                entity_key
            ):

                new_row[key_name] = key_value

            history = pd.concat(
                [
                    history,
                    pd.DataFrame([new_row])
                ],
                ignore_index=True
            )

    return pd.DataFrame(
        forecast_results
    )

In [861]:
# DIAGNOSTIC: Check why forecast_df is empty
print(f"model_df shape: {model_df.shape}")
print(f"model_df columns: {model_df.columns.tolist()}")
print(f"\nmodel_df head:")
print(model_df.head())

print(f"\n\nUnique values per key:")
for key in FORECAST_KEYS:
    if key in model_df.columns:
        print(f"  {key}: {model_df[key].nunique()} unique values")

print(f"\nMinimum history months required: {FORECAST_CONFIG.minimum_history_months}")
print(f"Date column: {CONFIG.date_col}")
print(f"Target column: {CONFIG.target_col}")

# Check history for each entity
print(f"\n\nHistory analysis per entity:")
for key_tuple in model_df.groupby(FORECAST_KEYS).groups.keys():
    subset = model_df[model_df[FORECAST_KEYS].eq(pd.Series(dict(zip(FORECAST_KEYS, key_tuple)))).all(axis=1)]
    months = subset[CONFIG.date_col].nunique()
    print(f"  {key_tuple}: {months} months, {len(subset)} rows")


model_df shape: (12, 62)
model_df columns: ['YEAR_MONTH', 'SPSR_ID', 'SBSR_ID', 'AMOUNT', 'SUMMARY_TYPE', 'YEAR', 'MONTH', 'MONTH_NUMBER', 'QUARTER', 'QUARTER_NUMBER', 'IS_YEAR_START', 'IS_YEAR_END', 'IS_QUARTER_START', 'IS_QUARTER_END', 'MONTH_SIN', 'MONTH_COS', 'QUARTER_SIN', 'QUARTER_COS', 'LAG_1', 'LAG_2', 'LAG_3', 'LAG_4', 'LAG_6', '_HISTORICAL_OBSERVATIONS', 'ROLLING_MEAN_2', 'ROLLING_STD_2', 'ROLLING_MIN_2', 'ROLLING_MAX_2', 'ROLLING_MEAN_3', 'ROLLING_STD_3', 'ROLLING_MIN_3', 'ROLLING_MAX_3', 'ROLLING_MEAN_6', 'ROLLING_STD_6', 'ROLLING_MIN_6', 'ROLLING_MAX_6', 'ROLLING_MEAN_12', 'ROLLING_STD_12', 'ROLLING_MIN_12', 'ROLLING_MAX_12', 'ROLLING_MEDIAN_2', 'ROLLING_MEDIAN_3', 'ROLLING_MEDIAN_6', 'ROLLING_MEDIAN_12', 'MOM_CHANGE', 'MOM_GROWTH_RATE', 'CHANGE_3M', 'GROWTH_3M', 'DEVIATION_FROM_MEAN_3', 'TREND_3M', 'TREND_6M', 'SUBSCRIBER_HIST_MEAN', 'SUBSCRIBER_HIST_MEDIAN', 'SUBSCRIBER_HIST_STD', 'SPONSOR_HIST_MEAN', 'SPONSOR_HIST_MEDIAN', 'SPONSOR_HIST_STD', 'ENTITY_HIST_MEAN', 'ENTITY

In [862]:
# DIAGNOSTIC: Check why forecast_df is empty
print(f"model_df shape: {model_df.shape}")
print(f"model_df columns: {model_df.columns.tolist()}")
print(f"\nmodel_df head:")
print(model_df.head())

print(f"\n\nUnique values per key:")
for key in FORECAST_KEYS:
    if key in model_df.columns:
        print(f"  {key}: {model_df[key].nunique()} unique values")

print(f"\nMinimum history months required: {FORECAST_CONFIG.minimum_history_months}")
print(f"Date column: {CONFIG.date_col}")
print(f"Target column: {CONFIG.target_col}")

# Check history for each entity
print(f"\n\nHistory analysis per entity:")
for key_tuple in model_df.groupby(FORECAST_KEYS).groups.keys():
    subset = model_df[model_df[FORECAST_KEYS].eq(pd.Series(dict(zip(FORECAST_KEYS, key_tuple)))).all(axis=1)]
    months = subset[CONFIG.date_col].nunique()
    print(f"  {key_tuple}: {months} months, {len(subset)} rows")

model_df shape: (12, 62)
model_df columns: ['YEAR_MONTH', 'SPSR_ID', 'SBSR_ID', 'AMOUNT', 'SUMMARY_TYPE', 'YEAR', 'MONTH', 'MONTH_NUMBER', 'QUARTER', 'QUARTER_NUMBER', 'IS_YEAR_START', 'IS_YEAR_END', 'IS_QUARTER_START', 'IS_QUARTER_END', 'MONTH_SIN', 'MONTH_COS', 'QUARTER_SIN', 'QUARTER_COS', 'LAG_1', 'LAG_2', 'LAG_3', 'LAG_4', 'LAG_6', '_HISTORICAL_OBSERVATIONS', 'ROLLING_MEAN_2', 'ROLLING_STD_2', 'ROLLING_MIN_2', 'ROLLING_MAX_2', 'ROLLING_MEAN_3', 'ROLLING_STD_3', 'ROLLING_MIN_3', 'ROLLING_MAX_3', 'ROLLING_MEAN_6', 'ROLLING_STD_6', 'ROLLING_MIN_6', 'ROLLING_MAX_6', 'ROLLING_MEAN_12', 'ROLLING_STD_12', 'ROLLING_MIN_12', 'ROLLING_MAX_12', 'ROLLING_MEDIAN_2', 'ROLLING_MEDIAN_3', 'ROLLING_MEDIAN_6', 'ROLLING_MEDIAN_12', 'MOM_CHANGE', 'MOM_GROWTH_RATE', 'CHANGE_3M', 'GROWTH_3M', 'DEVIATION_FROM_MEAN_3', 'TREND_3M', 'TREND_6M', 'SUBSCRIBER_HIST_MEAN', 'SUBSCRIBER_HIST_MEDIAN', 'SUBSCRIBER_HIST_STD', 'SPONSOR_HIST_MEAN', 'SPONSOR_HIST_MEDIAN', 'SPONSOR_HIST_STD', 'ENTITY_HIST_MEAN', 'ENTITY

In [863]:
forecast_df = generate_recursive_forecasts(
    model=final_xgb_model,

    data=model_df,

    forecast_keys=FORECAST_KEYS,

    feature_columns=FEATURE_COLUMNS,

    config=CONFIG,

    forecast_config=FORECAST_CONFIG,

    target_transform=selected_target_transform
)

In [864]:
display(forecast_df )
print(final_xgb_model)

,SPSR_ID,SBSR_ID,SUMMARY_TYPE,FORECAST_MONTH,FORECAST_MONEY_OUT,MODEL_NAME,MODEL_VERSION,FORECAST_HORIZON,DATA_HISTORY_MONTHS,FORECAST_RUN_DATE
0,AG0025,AG0025S100000097,MoneyOut,2026-08-01,122343.921875,MONEY_OUT_XGBOOST,1.0.0,1,12,2026-08-31 01:46:52.064595+00:00
1,AG0025,AG0025S100000097,MoneyOut,2026-09-01,122343.921875,MONEY_OUT_XGBOOST,1.0.0,2,12,2026-08-31 01:46:52.079308+00:00


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8183562936796783, device=None,
             early_stopping_rounds=None, enable_categorical=True,
             eval_metric=None, feature_types=None, feature_weights=None,
             gamma=3.759064545366492, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.05160037849149582,
             max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=3, max_leaves=None,
             min_child_weight=1, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=111, n_jobs=-1,
             num_parallel_tree=None, ...)


In [865]:
display(
    forecast_df.sort_values(
        FORECAST_KEYS
        + ["FORECAST_MONTH"]
    )
)

,SPSR_ID,SBSR_ID,SUMMARY_TYPE,FORECAST_MONTH,FORECAST_MONEY_OUT,MODEL_NAME,MODEL_VERSION,FORECAST_HORIZON,DATA_HISTORY_MONTHS,FORECAST_RUN_DATE
0,AG0025,AG0025S100000097,MoneyOut,2026-08-01,122343.921875,MONEY_OUT_XGBOOST,1.0.0,1,12,2026-08-31 01:46:52.064595+00:00
1,AG0025,AG0025S100000097,MoneyOut,2026-09-01,122343.921875,MONEY_OUT_XGBOOST,1.0.0,2,12,2026-08-31 01:46:52.079308+00:00


In [866]:
#forecast_df = model_df.sort_values(by=sort_cols)

display(forecast_df)

,SPSR_ID,SBSR_ID,SUMMARY_TYPE,FORECAST_MONTH,FORECAST_MONEY_OUT,MODEL_NAME,MODEL_VERSION,FORECAST_HORIZON,DATA_HISTORY_MONTHS,FORECAST_RUN_DATE
0,AG0025,AG0025S100000097,MoneyOut,2026-08-01,122343.921875,MONEY_OUT_XGBOOST,1.0.0,1,12,2026-08-31 01:46:52.064595+00:00
1,AG0025,AG0025S100000097,MoneyOut,2026-09-01,122343.921875,MONEY_OUT_XGBOOST,1.0.0,2,12,2026-08-31 01:46:52.079308+00:00


In [867]:
expected_forecast_months = set(
    forecast_months
)

if forecast_df.empty:
    print("WARNING: forecast_df is empty. No valid forecasts generated.")
    print(f"Expected forecast months: {expected_forecast_months}")
else:
    actual_forecast_months = set(
        forecast_df[
            "FORECAST_MONTH"
        ]
    )

    if actual_forecast_months != expected_forecast_months:

        raise ValueError(
            "Forecast months do not match "
            "the requested horizon."
        )

    print(
        "Forecast horizon validation passed."
    )

Forecast horizon validation passed.


In [868]:
def validate_forecast_cutoff(
    historical_data: pd.DataFrame,
    forecast_data: pd.DataFrame,
    date_column: str,
    forecast_month_column: str
) -> None:
    """
    Ensure forecasts are strictly after the historical cutoff.
    """

    historical_max = (
        pd.to_datetime(
            historical_data[date_column]
        ).max()
    )

    forecast_min = (
        pd.to_datetime(
            forecast_data[
                forecast_month_column
            ]
        ).min()
    )

    if forecast_min <= historical_max:

        raise ValueError(
            "Forecast contains dates at or before "
            "the historical cutoff."
        )

    logger.info(
        "Forecast cutoff validation passed."
    )

In [869]:
validate_forecast_cutoff(
    historical_data=model_df,
    forecast_data=forecast_df,
    date_column=CONFIG.date_col,
    forecast_month_column="FORECAST_MONTH"
)

2026-08-30 21:46:52,287 | INFO | Forecast cutoff validation passed.


In [870]:
negative_predictions = forecast_df[
    forecast_df[
        "FORECAST_MONEY_OUT"
    ] < 0
]

if not negative_predictions.empty:

    raise ValueError(
        "Negative MONEY_OUT forecasts detected."
    )

In [871]:
eligible_entities = []

for entity_key, history in forecast_state.items():

    history_count = (
        history[
            CONFIG.target_col
        ]
        .notna()
        .sum()
    )

    if (
        history_count
        >= FORECAST_CONFIG.minimum_history_months
    ):

        eligible_entities.append(
            entity_key
        )

expected_rows = (
    len(eligible_entities)
    * FORECAST_CONFIG.horizon_months
)

actual_rows = len(
    forecast_df
)

print(
    "Expected forecast rows:",
    expected_rows
)

print(
    "Actual forecast rows:",
    actual_rows
)

Expected forecast rows: 2
Actual forecast rows: 2


In [872]:
final_validation_wape = (
    optimized_window_results_df[
        "WAPE"
    ].mean()
)

In [873]:
forecast_df[
    "MODEL_VALIDATION_WAPE"
] = final_validation_wape

In [876]:
forecast_df = forecast_df[
    [
        "SPSR_ID",
        "SBSR_ID",
        "SUMMARY_TYPE",
        "FORECAST_MONTH",
        "FORECAST_MONEY_OUT",
        "MODEL_NAME",
        "MODEL_VERSION",
        "FORECAST_HORIZON",
        "DATA_HISTORY_MONTHS",
        "MODEL_VALIDATION_WAPE",
        "FORECAST_RUN_DATE"
    ]
]

In [878]:
display(
    forecast_df.sort_values(
        [
            "SPSR_ID",
            "SBSR_ID",
            "FORECAST_MONTH"
        ]
    )
)

,SPSR_ID,SBSR_ID,SUMMARY_TYPE,FORECAST_MONTH,FORECAST_MONEY_OUT,MODEL_NAME,MODEL_VERSION,FORECAST_HORIZON,DATA_HISTORY_MONTHS,MODEL_VALIDATION_WAPE,FORECAST_RUN_DATE
0,AG0025,AG0025S100000097,MoneyOut,2026-08-01,122343.921875,MONEY_OUT_XGBOOST,1.0.0,1,12,0.208244,2026-08-31 01:46:52.064595+00:00
1,AG0025,AG0025S100000097,MoneyOut,2026-09-01,122343.921875,MONEY_OUT_XGBOOST,1.0.0,2,12,0.208244,2026-08-31 01:46:52.079308+00:00


#           Working model

In [909]:
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd

from xgboost import XGBRegressor


@dataclass
class ForecastConfig:

    entity_columns: List[str]

    date_column: str = "YEAR_MONTH"

    target_column: str = "AMOUNT"

    forecast_horizon: int = 2

    random_seed: int = 42

    min_xgb_history: int = 6

    min_trend_history: int = 3

    xgb_params: Dict = None

    def __post_init__(self):

        if self.xgb_params is None:

            self.xgb_params = {
                "n_estimators": 300,
                "learning_rate": 0.03,
                "max_depth": 3,
                "min_child_weight": 3,
                "subsample": 0.8,
                "colsample_bytree": 0.8,
                "reg_alpha": 0.1,
                "reg_lambda": 1.0,
                "objective": "reg:squarederror",
                "random_state": self.random_seed,
                "n_jobs": -1
            }

In [910]:
CONFIG = ForecastConfig(
    entity_columns=[
        "SPSR_ID",
        "SBSR_ID",
        "SUMMARY_TYPE"
    ],
    date_column="YEAR_MONTH",
    target_column="AMOUNT"
)

In [911]:
def prepare_source_data(
    df: pd.DataFrame,
    config: ForecastConfig
) -> pd.DataFrame:

    data = df.copy()

    required = (
        config.entity_columns
        + [
            config.date_column,
            config.target_column
        ]
    )

    missing = [
        c for c in required
        if c not in data.columns
    ]

    if missing:
        raise ValueError(
            f"Missing columns: {missing}"
        )

    data[config.date_column] = (
        pd.to_datetime(
            data[config.date_column]
        )
        .dt.to_period("M")
        .dt.to_timestamp()
    )

    data[config.target_column] = (
        pd.to_numeric(
            data[config.target_column],
            errors="coerce"
        )
    )

    data = data.dropna(
        subset=[
            config.date_column,
            config.target_column
        ]
    )

    data = (
        data
        .sort_values(
            config.entity_columns
            + [config.date_column]
        )
        .reset_index(drop=True)
    )

    return data

In [912]:
def validate_duplicates(
    df: pd.DataFrame,
    config: ForecastConfig
) -> pd.DataFrame:

    keys = (
        config.entity_columns
        + [config.date_column]
    )

    duplicates = (
        df
        .groupby(keys)
        .size()
        .reset_index(
            name="ROW_COUNT"
        )
    )

    duplicates = duplicates[
        duplicates["ROW_COUNT"] > 1
    ]

    if not duplicates.empty:

        print(
            "WARNING: Duplicate monthly records found."
        )

    return duplicates

In [913]:
data = {
    "SPSR_ID": [
        "AG0025"
    ] * 12,

    "SBSR_ID": [
        "AG0025S100000097"
    ] * 12,

    "YEAR_MONTH": [
        "2025-08-01",
        "2025-09-01",
        "2025-10-01",
        "2025-11-01",
        "2025-12-01",
        "2026-01-01",
        "2026-02-01",
        "2026-03-01",
        "2026-04-01",
        "2026-05-01",
        "2026-06-01",
        "2026-07-01"
    ],

    "AMOUNT": [
        513.00,
        19268.75,
        28054.06,
        18634.24,
        32127.32,
        42061.26,
        55587.25,
        68903.75,
        82319.75,
        95735.75,
        109151.75,
        122567.75
    ],

    "SUMMARY_TYPE": [
        "MoneyOut"
    ] * 12
}

source_df = pd.DataFrame(data)

In [914]:
duplicates = validate_duplicates(
    source_df,
    CONFIG
)

display(duplicates)

,SPSR_ID,SBSR_ID,SUMMARY_TYPE,YEAR_MONTH,ROW_COUNT


In [915]:
def aggregate_monthly_data(
    df: pd.DataFrame,
    config: ForecastConfig
) -> pd.DataFrame:

    group_columns = (
        config.entity_columns
        + [config.date_column]
    )

    result = (
        df
        .groupby(
            group_columns,
            as_index=False
        )[config.target_column]
        .sum()
    )

    return (
        result
        .sort_values(
            config.entity_columns
            + [config.date_column]
        )
        .reset_index(drop=True)
    )

In [916]:
def create_model_features(
    df: pd.DataFrame,
    config: ForecastConfig
) -> pd.DataFrame:

    data = df.copy()

    group = (
        data
        .groupby(
            config.entity_columns,
            dropna=False
        )[config.target_column]
    )

    # -----------------------------
    # Lags
    # -----------------------------

    for lag in [1, 2, 3, 4, 6, 12]:

        data[f"LAG_{lag}"] = (
            group.shift(lag)
        )

    # -----------------------------
    # Rolling features
    # IMPORTANT:
    # shift(1) BEFORE rolling
    # -----------------------------

    shifted = group.shift(1)

    for window in [3, 6, 12]:

        rolling_group = (
            shifted
            .groupby(
                [
                    data[c]
                    for c in config.entity_columns
                ],
                dropna=False
            )
        )

        data[
            f"ROLLING_MEAN_{window}"
        ] = (
            rolling_group
            .rolling(
                window,
                min_periods=1
            )
            .mean()
            .reset_index(
                level=config.entity_columns,
                drop=True
            )
        )

        data[
            f"ROLLING_STD_{window}"
        ] = (
            rolling_group
            .rolling(
                window,
                min_periods=2
            )
            .std()
            .reset_index(
                level=config.entity_columns,
                drop=True
            )
        )

    # -----------------------------
    # Calendar
    # -----------------------------

    dates = pd.to_datetime(
        data[config.date_column]
    )

    data["YEAR"] = dates.dt.year

    data["MONTH"] = dates.dt.month

    data["QUARTER"] = dates.dt.quarter

    data["MONTH_SIN"] = np.sin(
        2 * np.pi * data["MONTH"] / 12
    )

    data["MONTH_COS"] = np.cos(
        2 * np.pi * data["MONTH"] / 12
    )

    # -----------------------------
    # Momentum
    # -----------------------------

    data["MOM_CHANGE"] = (
        data["LAG_1"]
        -
        data["LAG_2"]
    )

    data["MOM_GROWTH_RATE"] = np.where(
        data["LAG_2"].abs() > 0,
        (
            data["LAG_1"]
            -
            data["LAG_2"]
        )
        /
        data["LAG_2"].abs(),
        0.0
    )

    # -----------------------------
    # Trend
    # -----------------------------

    data["TREND_3M"] = (
        data["LAG_1"]
        -
        data["LAG_3"]
    ) / 2.0

    data["TREND_6M"] = (
        data["LAG_1"]
        -
        data["LAG_6"]
    ) / 5.0

    return data

In [917]:
def select_training_rows(
    feature_df: pd.DataFrame
) -> pd.DataFrame:

    required = [
        "LAG_1",
        "LAG_2",
        "LAG_3"
    ]

    result = feature_df.dropna(
        subset=required
    ).copy()

    return result

In [918]:
XGB_FEATURES = [
    "LAG_1",
    "LAG_2",
    "LAG_3",
    "LAG_4",
    "LAG_6",
    "LAG_12",

    "ROLLING_MEAN_3",
    "ROLLING_MEAN_6",
    "ROLLING_MEAN_12",

    "ROLLING_STD_3",
    "ROLLING_STD_6",
    "ROLLING_STD_12",

    "MOM_CHANGE",
    "MOM_GROWTH_RATE",

    "TREND_3M",
    "TREND_6M",

    "YEAR",
    "MONTH",
    "QUARTER",

    "MONTH_SIN",
    "MONTH_COS"
]

In [919]:
def get_available_features(
    df: pd.DataFrame,
    requested_features: List[str]
) -> List[str]:

    available = []

    for feature in requested_features:

        if feature not in df.columns:
            continue

        if df[feature].notna().sum() == 0:
            continue

        available.append(feature)

    return available

In [920]:
def train_xgboost(
    training_df: pd.DataFrame,
    config: ForecastConfig
) -> Tuple[XGBRegressor, List[str]]:

    features = get_available_features(
        training_df,
        XGB_FEATURES
    )

    # Remove extremely sparse features
    usable_features = []

    for feature in features:

        non_null_ratio = (
            training_df[feature]
            .notna()
            .mean()
        )

        if non_null_ratio >= 0.50:

            usable_features.append(
                feature
            )

    features = usable_features

    if len(features) == 0:

        raise ValueError(
            "No usable XGBoost features."
        )

    X = (
        training_df[features]
        .replace(
            [np.inf, -np.inf],
            np.nan
        )
        .fillna(0)
    )

    y = training_df[
        config.target_column
    ]

    model = XGBRegressor(
        **config.xgb_params
    )

    model.fit(
        X,
        y,
        verbose=False
    )

    return model, features

In [921]:
def linear_trend_forecast(
    history: pd.DataFrame,
    config: ForecastConfig,
    periods: int = 2
) -> List[float]:

    y = (
        history[
            config.target_column
        ]
        .astype(float)
        .values
    )

    if len(y) < 3:

        last_value = float(y[-1])

        return [
            last_value
            for _ in range(periods)
        ]

    x = np.arange(
        len(y)
    )

    # Use recent history.
    # For short datasets use all history.
    window = min(
        len(y),
        6
    )

    x_recent = x[-window:]

    y_recent = y[-window:]

    slope, intercept = np.polyfit(
        x_recent,
        y_recent,
        1
    )

    future_x = np.arange(
        len(y),
        len(y) + periods
    )

    predictions = (
        slope * future_x
        + intercept
    )

    return [
        max(
            0.0,
            float(value)
        )
        for value in predictions
    ]

In [922]:
def naive_forecast(
    history: pd.DataFrame,
    config: ForecastConfig,
    periods: int = 2
) -> List[float]:

    last_value = float(
        history[
            config.target_column
        ].iloc[-1]
    )

    return [
        max(0.0, last_value)
        for _ in range(periods)
    ]

In [923]:
def create_future_feature_row(
    history: pd.DataFrame,
    forecast_month: pd.Timestamp,
    config: ForecastConfig,
    feature_columns: List[str]
) -> pd.DataFrame:

    history = (
        history
        .sort_values(
            config.date_column
        )
        .reset_index(drop=True)
    )

    values = (
        history[
            config.target_column
        ]
        .astype(float)
        .tolist()
    )

    row = {}

    # -----------------------------
    # Calendar
    # -----------------------------

    row["YEAR"] = forecast_month.year

    row["MONTH"] = forecast_month.month

    row["QUARTER"] = (
        forecast_month.quarter
    )

    row["MONTH_SIN"] = np.sin(
        2 * np.pi
        * forecast_month.month
        / 12
    )

    row["MONTH_COS"] = np.cos(
        2 * np.pi
        * forecast_month.month
        / 12
    )

    # -----------------------------
    # Lags
    # -----------------------------

    for lag in [1, 2, 3, 4, 6, 12]:

        if len(values) >= lag:

            row[
                f"LAG_{lag}"
            ] = values[-lag]

        else:

            row[
                f"LAG_{lag}"
            ] = np.nan

    # -----------------------------
    # Rolling
    # -----------------------------

    for window in [3, 6, 12]:

        recent = values[-window:]

        if recent:

            row[
                f"ROLLING_MEAN_{window}"
            ] = np.mean(recent)

            row[
                f"ROLLING_STD_{window}"
            ] = (
                np.std(recent, ddof=1)
                if len(recent) > 1
                else 0.0
            )

        else:

            row[
                f"ROLLING_MEAN_{window}"
            ] = np.nan

            row[
                f"ROLLING_STD_{window}"
            ] = np.nan

    # -----------------------------
    # Momentum
    # -----------------------------

    if len(values) >= 2:

        row["MOM_CHANGE"] = (
            values[-1]
            -
            values[-2]
        )

        if values[-2] != 0:

            row["MOM_GROWTH_RATE"] = (
                values[-1]
                -
                values[-2]
            ) / abs(values[-2])

        else:

            row["MOM_GROWTH_RATE"] = 0.0

    else:

        row["MOM_CHANGE"] = 0.0

        row["MOM_GROWTH_RATE"] = 0.0

    # -----------------------------
    # Trend
    # -----------------------------

    if len(values) >= 3:

        row["TREND_3M"] = (
            values[-1]
            -
            values[-3]
        ) / 2.0

    else:

        row["TREND_3M"] = 0.0

    if len(values) >= 6:

        row["TREND_6M"] = (
            values[-1]
            -
            values[-6]
        ) / 5.0

    else:

        row["TREND_6M"] = 0.0

    result = pd.DataFrame(
        [row]
    )

    return result.reindex(
        columns=feature_columns,
        fill_value=np.nan
    )

In [924]:
def recursive_xgboost_forecast(
    history: pd.DataFrame,
    model: XGBRegressor,
    feature_columns: List[str],
    config: ForecastConfig,
    periods: int = 2
) -> List[float]:

    working_history = (
        history
        .sort_values(
            config.date_column
        )
        .copy()
        .reset_index(drop=True)
    )

    latest_month = (
        working_history[
            config.date_column
        ].max()
    )

    predictions = []

    for step in range(1, periods + 1):

        forecast_month = (
            latest_month
            +
            pd.DateOffset(
                months=step
            )
        )

        future_features = (
            create_future_feature_row(
                working_history,
                forecast_month,
                config,
                feature_columns
            )
        )

        X_future = (
            future_features
            .replace(
                [np.inf, -np.inf],
                np.nan
            )
            .fillna(0)
        )

        prediction = float(
            model.predict(
                X_future
            )[0]
        )

        prediction = max(
            0.0,
            prediction
        )

        predictions.append(
            prediction
        )

        # ==================================
        # CRITICAL
        # Add prediction to history
        # ==================================

        new_row = {}

        for column in working_history.columns:

            if column in config.entity_columns:

                new_row[column] = (
                    working_history[
                        column
                    ].iloc[-1]
                )

            elif (
                column
                == config.date_column
            ):

                new_row[column] = (
                    forecast_month
                )

            elif (
                column
                == config.target_column
            ):

                new_row[column] = (
                    prediction
                )

            else:

                new_row[column] = np.nan

        working_history = pd.concat(
            [
                working_history,
                pd.DataFrame([new_row])
            ],
            ignore_index=True
        )

    return predictions

In [925]:
def evaluate_xgboost_walk_forward(
    history: pd.DataFrame,
    config: ForecastConfig
) -> Dict[str, float]:

    history = (
        history
        .sort_values(
            config.date_column
        )
        .reset_index(drop=True)
    )

    if len(history) < 7:

        return {
            "MAE": np.inf,
            "RMSE": np.inf
        }

    actuals = []

    predictions = []

    start = max(
        4,
        len(history) - 5
    )

    for test_index in range(
        start,
        len(history)
    ):

        train = history.iloc[
            :test_index
        ].copy()

        actual = float(
            history[
                config.target_column
            ].iloc[test_index]
        )

        features = create_model_features(
            train,
            config
        )

        train_features = (
            select_training_rows(
                features
            )
        )

        if len(train_features) < 3:
            continue

        try:

            model, feature_columns = (
                train_xgboost(
                    train_features,
                    config
                )
            )

            forecast_month = (
                history[
                    config.date_column
                ].iloc[test_index]
            )

            future_features = (
                create_future_feature_row(
                    train,
                    forecast_month,
                    config,
                    feature_columns
                )
            )

            prediction = float(
                model.predict(
                    future_features
                    .fillna(0)
                )[0]
            )

            actuals.append(actual)

            predictions.append(
                prediction
            )

        except Exception:

            continue

    if not predictions:

        return {
            "MAE": np.inf,
            "RMSE": np.inf
        }

    errors = (
        np.array(predictions)
        -
        np.array(actuals)
    )

    mae = np.mean(
        np.abs(errors)
    )

    rmse = np.sqrt(
        np.mean(
            errors ** 2
        )
    )

    return {
        "MAE": float(mae),
        "RMSE": float(rmse)
    }

In [926]:
def evaluate_trend_walk_forward(
    history: pd.DataFrame,
    config: ForecastConfig
) -> Dict[str, float]:

    history = (
        history
        .sort_values(
            config.date_column
        )
        .reset_index(drop=True)
    )

    actuals = []

    predictions = []

    for test_index in range(
        3,
        len(history)
    ):

        train = history.iloc[
            :test_index
        ]

        actual = float(
            history[
                config.target_column
            ].iloc[test_index]
        )

        prediction = linear_trend_forecast(
            train,
            config,
            periods=1
        )[0]

        actuals.append(actual)

        predictions.append(
            prediction
        )

    errors = (
        np.array(predictions)
        -
        np.array(actuals)
    )

    return {
        "MAE": float(
            np.mean(
                np.abs(errors)
            )
        ),

        "RMSE": float(
            np.sqrt(
                np.mean(
                    errors ** 2
                )
            )
        )
    }

In [927]:
def select_best_model(
    history: pd.DataFrame,
    config: ForecastConfig
) -> str:

    results = {}

    # -----------------------------
    # Naive
    # -----------------------------

    actuals = history[
        config.target_column
    ].iloc[1:].values

    naive_predictions = (
        history[
            config.target_column
        ].shift(1)
        .iloc[1:]
        .values
    )

    naive_mae = np.mean(
        np.abs(
            actuals
            -
            naive_predictions
        )
    )

    results["NAIVE"] = naive_mae

    # -----------------------------
    # Trend
    # -----------------------------

    trend_metrics = (
        evaluate_trend_walk_forward(
            history,
            config
        )
    )

    results["LINEAR_TREND"] = (
        trend_metrics["MAE"]
    )

    # -----------------------------
    # XGBoost
    # -----------------------------

    if len(history) >= config.min_xgb_history:

        xgb_metrics = (
            evaluate_xgboost_walk_forward(
                history,
                config
            )
        )

        results["XGBOOST"] = (
            xgb_metrics["MAE"]
        )

    best_model = min(
        results,
        key=results.get
    )

    print(
        "Model validation results:"
    )

    for model_name, mae in results.items():

        print(
            f"{model_name:15s} "
            f"MAE = {mae:,.2f}"
        )

    print(
        f"\nSelected model: {best_model}"
    )

    return best_model

In [928]:
def forecast_entity(
    history: pd.DataFrame,
    config: ForecastConfig
) -> pd.DataFrame:

    history = (
        history
        .sort_values(
            config.date_column
        )
        .reset_index(drop=True)
    )

    latest_month = (
        history[
            config.date_column
        ].max()
    )

    best_model = select_best_model(
        history,
        config
    )

    # ==================================
    # NAIVE
    # ==================================

    if best_model == "NAIVE":

        predictions = naive_forecast(
            history,
            config,
            config.forecast_horizon
        )

    # ==================================
    # LINEAR TREND
    # ==================================

    elif best_model == "LINEAR_TREND":

        predictions = linear_trend_forecast(
            history,
            config,
            config.forecast_horizon
        )

    # ==================================
    # XGBOOST
    # ==================================

    elif best_model == "XGBOOST":

        feature_df = create_model_features(
            history,
            config
        )

        train_df = select_training_rows(
            feature_df
        )

        model, feature_columns = (
            train_xgboost(
                train_df,
                config
            )
        )

        predictions = (
            recursive_xgboost_forecast(
                history,
                model,
                feature_columns,
                config,
                config.forecast_horizon
            )
        )

    else:

        raise ValueError(
            f"Unknown model: {best_model}"
        )

    # ==================================
    # Build output
    # ==================================

    output = []

    for horizon, prediction in enumerate(
        predictions,
        start=1
    ):

        forecast_month = (
            latest_month
            +
            pd.DateOffset(
                months=horizon
            )
        )

        row = {}

        for column in config.entity_columns:

            row[column] = (
                history[
                    column
                ].iloc[-1]
            )

        row["FORECAST_MONTH"] = (
            forecast_month
        )

        row["FORECAST_MONEY_OUT"] = (
            prediction
        )

        row["MODEL_NAME"] = (
            best_model
        )

        row["FORECAST_HORIZON"] = (
            horizon
        )

        row["DATA_HISTORY_MONTHS"] = (
            len(history)
        )

        output.append(row)

    return pd.DataFrame(output)

In [929]:
source_df = prepare_source_data(
    source_df,
    CONFIG
)

source_df = aggregate_monthly_data(
    source_df,
    CONFIG
)

In [930]:
entity_history = source_df.copy()

forecast_df = forecast_entity(
    entity_history,
    CONFIG
)

display(forecast_df)

Model validation results:
NAIVE           MAE = 12,808.58
LINEAR_TREND    MAE = 6,818.54
XGBOOST         MAE = 28,322.75

Selected model: LINEAR_TREND


,SPSR_ID,SBSR_ID,SUMMARY_TYPE,FORECAST_MONTH,FORECAST_MONEY_OUT,MODEL_NAME,FORECAST_HORIZON,DATA_HISTORY_MONTHS
0,AG0025,AG0025S100000097,MoneyOut,2026-08-01,135950.583333,LINEAR_TREND,1,12
1,AG0025,AG0025S100000097,MoneyOut,2026-09-01,149352.369048,LINEAR_TREND,2,12
